In [24]:
!pip install python-telegram-bot requests nest_asyncio

In [25]:
!pip install pyngrok

In [26]:
from pyngrok import ngrok

ngrok.set_auth_token("3FDJVjVBsVSMcpG8WkyNnSIGDHj_2HEAmTr7Duf7CQENsekZY")

public_url = ngrok.connect(8000)

print(public_url)

NgrokTunnel: "https://area-extenuate-falsify.ngrok-free.dev" -> "http://localhost:8000"


In [27]:
!pip install nest_asyncio

In [28]:
import nest_asyncio
nest_asyncio.apply()

In [29]:
# ===== MTB API Bridge =====

from fastapi import FastAPI
import uvicorn
import threading

app = FastAPI()

LATEST_MTB_SIGNALS = []

@app.get("/api/v1/scanner/market-state")
async def market_state():
    return {
        "market": "UNKNOWN",
        "trend": "SIDEWAYS"
    }

@app.get("/api/v1/scanner/signals")
async def scanner_signals(strategy: str = "MTB"):
    if strategy != "MTB":
        return []
    return LATEST_MTB_SIGNALS


def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_api, daemon=True).start()

print("✅ MTB API Bridge Started")

✅ MTB API Bridge Started


/usr/local/lib/python3.12/dist-packages/uvicorn/server.py:75: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
Exception in thread Thread-9 (run_api):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_114072/599496640.py", line 26, in run_api
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 620, in run
    server.run()
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 75, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: _patch_asyncio.<locals>.run() got an unexpected keyword arg

In [30]:
from __future__ import annotations

import asyncio
import nest_asyncio  # allows asyncio in Jupyter/Colab environments
nest_asyncio.apply()
import json
import logging
import os
import shutil
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Optional

import requests
from telegram import Update
from telegram.ext import Application, ApplicationBuilder, CommandHandler, ContextTypes


# =============================================================================
# CONFIGURATION
# =============================================================================
# Keep all values that a beginner may want to tune in one place.

from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

COLAB_DRIVE_DIR = Path("/content/drive/MyDrive/CryptoScanner")


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def initialize_storage_dir() -> Path:
    requested = os.getenv("CRYPTO_SCANNER_DIR")
    if requested:
        storage_dir = Path(requested)
        storage_dir.mkdir(parents=True, exist_ok=True)
        return storage_dir

    if running_in_colab():
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive", force_remount=False)
            COLAB_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
            return COLAB_DRIVE_DIR
        except Exception:
            print("Google Drive mount failed. Falling back to local Colab storage.")

    BASE_DIR.mkdir(parents=True, exist_ok=True)
    return BASE_DIR


STORAGE_DIR = initialize_storage_dir()
USING_GOOGLE_DRIVE = str(STORAGE_DIR).startswith(str(COLAB_DRIVE_DIR))


def backup_file(path: Path) -> None:
    if not path.exists():
        return
    backup_path = path.with_suffix(path.suffix + ".bak")
    try:
        shutil.copy2(path, backup_path)
    except OSError:
        pass


def write_json_safely(path: Path, data) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    backup_file(path)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    temp_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
    temp_path.replace(path)


def migrate_file_to_storage(local_name: str, storage_name: str) -> None:
    destination = STORAGE_DIR / storage_name
    if destination.exists():
        return

    candidates = [
        BASE_DIR / local_name,
        BASE_DIR / storage_name,
        Path.cwd() / local_name,
        Path.cwd() / storage_name,
    ]
    for candidate in candidates:
        try:
            if candidate.exists() and candidate.resolve() != destination.resolve():
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(candidate, destination)
                return
        except OSError:
            continue


DEFAULT_WATCHLIST = ["BTC", "SOL", "ETH", "ZEC", "XRP", "BNB"]
def ensure_storage_files() -> None:
    migrate_file_to_storage("watchlist.json", "watchlist.json")
    migrate_file_to_storage("signal_performance.json", "signals.json")
    migrate_file_to_storage("signals.json", "signals.json")
    migrate_file_to_storage("stats.json", "stats.json")
    for name, default in {
        "watchlist.json": {"coins": DEFAULT_WATCHLIST},
        "signals.json": {"signals": []},
        "stats.json": {},
    }.items():
        path = STORAGE_DIR / name
        if not path.exists():
            write_json_safely(path, default)
    (STORAGE_DIR / "scanner.log").touch(exist_ok=True)
    if USING_GOOGLE_DRIVE:
        print(f"[CryptoScanner] Google Drive storage ready: {STORAGE_DIR}")
    else:
        print(f"[CryptoScanner] Local storage ready: {STORAGE_DIR}")
BOT_TOKEN ="8504132250:AAFVydqTqO12WWhlH45QL9dQ7Mu_2vnO4hc"

COINDCX_TICKER_URL    = "https://api.coindcx.com/exchange/ticker"
COINDCX_CANDLES_URL   = "https://public.coindcx.com/market_data/candles"
REQUEST_TIMEOUT_SECONDS = 10
TICKER_CACHE_TTL_SECONDS = int(os.getenv("TICKER_CACHE_TTL_SECONDS", "20"))

WATCHLIST_FILE = os.getenv("WATCHLIST_FILE", str(STORAGE_DIR / "watchlist.json"))
SIGNAL_LOG_FILE = os.getenv("SIGNAL_LOG_FILE", str(STORAGE_DIR / "signals.json"))
STATS_FILE = os.getenv("STATS_FILE", str(STORAGE_DIR / "stats.json"))
SCANNER_LOG_FILE = os.getenv("SCANNER_LOG_FILE", str(STORAGE_DIR / "scanner.log"))

# Run migration + create missing files in storage dir
ensure_storage_files()
QUOTE_PRIORITY = ("INR", "USDT")

SCAN_INTERVAL_SECONDS = int(os.getenv("SCAN_INTERVAL_SECONDS", "300"))
DISCOVERY_INTERVAL_SECONDS = int(os.getenv("DISCOVERY_INTERVAL_SECONDS", "900"))
DISCOVERY_MAX_COINS = int(os.getenv("DISCOVERY_MAX_COINS", "500"))
SCAN_CONCURRENCY = int(os.getenv("SCAN_CONCURRENCY", "50"))
BOOTSTRAP_CONCURRENCY = int(os.getenv("BOOTSTRAP_CONCURRENCY", "30"))  # parallel candle fetches
BOOTSTRAP_ENABLED = os.getenv("BOOTSTRAP_ENABLED", "true").lower() != "false"
# Raised from 100_000 to 500_000 to prioritise highly-traded coins
MIN_VOLUME_24H = float(os.getenv("MIN_VOLUME_24H", "500000"))
# Minimum liquidity proxy: quote volume (INR/USDT) over 24h
MIN_LIQUIDITY_24H = float(os.getenv("MIN_LIQUIDITY_24H", "1000000"))
MIN_PRICE = float(os.getenv("MIN_PRICE", "0.01"))
MAX_RESULTS = int(os.getenv("MAX_RESULTS", "10"))

ALERT_COOLDOWN_SECONDS = int(os.getenv("ALERT_COOLDOWN_SECONDS", "1800"))

# Telegram network timeouts — raised for Colab / slow connections
TG_CONNECT_TIMEOUT = float(os.getenv("TG_CONNECT_TIMEOUT", "30.0"))
TG_READ_TIMEOUT    = float(os.getenv("TG_READ_TIMEOUT",    "30.0"))
TG_WRITE_TIMEOUT   = float(os.getenv("TG_WRITE_TIMEOUT",   "30.0"))
TG_POOL_TIMEOUT    = float(os.getenv("TG_POOL_TIMEOUT",    "15.0"))

# Model version — bump this whenever signal logic changes so results
# can be compared across versions in /stats
MODEL_VERSION = "v12.2"

# COIN CLASSIFICATION — data collection only, no effect on signal logic
COIN_CLASSES: dict[str, set] = {
    "A": {"BTC", "ETH", "BNB", "SOL", "XRP"},
    "B": {"LINK", "AVAX", "SUI", "APT", "ARB", "NEAR", "INJ", "RENDER"},
}


def get_coin_class(symbol: str) -> str:
    sym = symbol.upper().split("-")[-1].split("_")[0]
    if sym in COIN_CLASSES["A"]:
        return "A"
    if sym in COIN_CLASSES["B"]:
        return "B"
    return "C"


EMA_FAST_PERIOD = 9
EMA_SLOW_PERIOD = 21
PRICE_HISTORY_LIMIT = 120

# MTF window sizes must be > EMA_FAST_PERIOD (9) so that
# min(EMA_FAST, window) < min(EMA_SLOW, window) — otherwise both EMAs
# collapse to the same period and fast > slow is always False.
#   5m  : 10 ticks (~50 min at 5-min scan interval)
#   15m : 24 ticks (~2 h)
#   1h  : 48 ticks (~4 h, full EMA_SLOW_PERIOD fits comfortably)
MTF_5M_WINDOW  = int(os.getenv("MTF_5M_WINDOW",  "10"))  # ticks for 5-min frame
MTF_15M_WINDOW = int(os.getenv("MTF_15M_WINDOW", "24"))  # ticks for 15-min frame
MTF_1H_WINDOW  = int(os.getenv("MTF_1H_WINDOW",  "48"))  # ticks for 1-hour frame

MOMENTUM_THRESHOLD_PERCENT = 3.0
VOLUME_SPIKE_MULTIPLIER = 2.0
VOLUME_AVERAGE_PERIOD = 20
VOLATILITY_LOOKBACK = 20
VOLATILITY_SPIKE_MULTIPLIER = 1.8

EVALUATION_HORIZONS = {
    "1h": 60 * 60,
    "4h": 4 * 60 * 60,
    "24h": 24 * 60 * 60,
}


# =============================================================================
# STORAGE STARTUP BANNER
# =============================================================================

_drive_label = "Google Drive" if USING_GOOGLE_DRIVE else "local disk"
print(f"[CryptoScanner] Storage: {STORAGE_DIR}  ({_drive_label})")

# =============================================================================
# LOGGING
# =============================================================================

logging.basicConfig(
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    level=logging.INFO,
)
logger = logging.getLogger("scanner_bot")
scanner_file_handler = logging.FileHandler(SCANNER_LOG_FILE, encoding="utf-8")
scanner_file_handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s: %(message)s"))
logger.addHandler(scanner_file_handler)


# =============================================================================
# WATCHLIST STORAGE
# =============================================================================
# The watchlist is saved as JSON so coins survive bot restarts.

class WatchlistStore:
    def __init__(self, path: str = WATCHLIST_FILE):
        self.path = Path(path)
        self._coins = self._load()

    def _load(self) -> list[str]:
        if not self.path.exists():
            return list(DEFAULT_WATCHLIST)

        try:
            data = json.loads(self.path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            logger.warning("Could not read watchlist file; using defaults", exc_info=True)
            return list(DEFAULT_WATCHLIST)

        coins = data.get("coins", data if isinstance(data, list) else [])
        normalized = [str(coin).upper().strip() for coin in coins if str(coin).strip()]
        return list(dict.fromkeys(normalized)) or list(DEFAULT_WATCHLIST)

    def save(self) -> None:
        write_json_safely(self.path, {"coins": self._coins})

    def all(self) -> list[str]:
        return list(self._coins)

    def add(self, coin: str) -> bool:
        normalized = coin.upper().strip()
        if not normalized or normalized in self._coins:
            return False
        self._coins.append(normalized)
        self.save()
        return True

    def remove(self, coin: str) -> bool:
        normalized = coin.upper().strip()
        if normalized not in self._coins:
            return False
        self._coins.remove(normalized)
        self.save()
        return True


# =============================================================================
# SIGNAL CALCULATION
# =============================================================================
# These functions only analyze market data. They never place orders.

@dataclass(frozen=True)
class Signal:
    coin: str
    kind: str
    score: int
    message: str
    price: float
    volume: float
    created_at: datetime
    tier: str
    reasons: list[str]
    volume_strength: float
    momentum_strength: float
    model_version: str = MODEL_VERSION
    # Phase 5 quality breakdown
    phase5_trend: int = 0
    phase5_pullback: int = 0
    phase5_momentum: int = 0
    phase5_risk_reward: int = 0
    phase5_total: int = 0
    final_score: int = 0
    # Historical Pattern Score breakdown
    hist_trend_7d:   int = 0
    hist_trend_30d:  int = 0
    hist_trend_90d:  int = 0
    hist_sr_quality: int = 0
    hist_vol_score:  int = 0
    hist_total:      int = 0
    coin_class:      str = "C"   # "A" | "B" | "C" — classification only
    market_state:    str = ""    # detected market condition
    opportunity_type: str = ""   # actionable opportunity classification
    opp_confidence:   int = 0    # 0-100 confidence in the opportunity type
    opportunity_score: int = 0   # 0-100 composite ranking score
    priority:          str = ""  # Elite | High | Medium | Watch | Ignore
    risk_level:        str = ""  # "low" | "medium" | "high"




# =============================================================================
# HISTORICAL PATTERN SCORE
# =============================================================================
# Uses CoinDCX public candle endpoint (no API key required).
# Endpoint: GET https://public.coindcx.com/market_data/candles
#   ?pair=B-BTC_USDT&resolution=1D&from=<epoch>&to=<epoch>
#
# Sub-scores (each 0-25):
#   trend_7d   – 7-day price trend direction and consistency
#   trend_30d  – 30-day trend
#   trend_90d  – 90-day trend
#   sr_quality – Support/resistance quality at current price
#   hist_vol   – Historical volatility regime (lower normalised vol = better)
#
# Final historical_score = clamp(sum, 0, 100)

# In-process cache: (market_pair → (fetched_at_epoch, candles_list))
_candle_cache: dict[str, tuple[float, list[dict]]] = {}
CANDLE_CACHE_TTL = 3600   # re-fetch daily candles at most once per hour


@dataclass(frozen=True)
class HistoricalPatternScore:
    trend_7d:   int   # 0-25
    trend_30d:  int   # 0-25
    trend_90d:  int   # 0-25
    sr_quality: int   # 0-25  (split across the four to keep total ≤ 100)
    hist_vol:   int   # 0-25  (penalises extreme volatility regimes)
    total:      int   # 0-100

    def as_dict(self) -> dict:
        return {
            "trend_7d":   self.trend_7d,
            "trend_30d":  self.trend_30d,
            "trend_90d":  self.trend_90d,
            "sr_quality": self.sr_quality,
            "hist_vol":   self.hist_vol,
            "total":      self.total,
        }


def _fetch_daily_candles(market_pair: str, days: int = 95) -> list[dict]:
    """Fetch daily OHLCV candles from CoinDCX public endpoint.

    Returns a list of dicts with keys: open, high, low, close, volume, time.
    Returns [] on any error so callers degrade gracefully.
    """
    import time as _time

    now = _time.time()
    cached = _candle_cache.get(market_pair)
    if cached and now - cached[0] < CANDLE_CACHE_TTL:
        return cached[1]

    to_ts   = int(now)
    from_ts = to_ts - days * 86400

    try:
        resp = requests.get(
            COINDCX_CANDLES_URL,
            params={"pair": market_pair, "resolution": "1D",
                    "from": from_ts, "to": to_ts},
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        resp.raise_for_status()
        data = resp.json()
        if isinstance(data, list) and data:
            _candle_cache[market_pair] = (now, data)
            return data
    except Exception:
        logger.debug("Candle fetch failed for %s", market_pair, exc_info=True)

    return []


def _coin_to_pair(coin: str) -> list[str]:
    """Map a coin symbol to candidate CoinDCX market pair strings."""
    coin = coin.upper()
    return [f"B-{coin}_USDT", f"B-{coin}_INR", f"B-{coin}_BTC"]


def _trend_score_from_closes(closes: list[float], max_pts: int = 25) -> int:
    """Score how cleanly *closes* trended upward.

    Combines:
    - net return over the window (up to 60 % gain = full marks)
    - percentage of up-days (consistency)
    """
    if len(closes) < 2:
        return 0
    net_ret   = percent_change(closes[0], closes[-1])
    ret_score = _clamp(net_ret / 60.0, -1.0, 1.0)           # 60 % gain = 1.0

    up_days   = sum(1 for i in range(1, len(closes)) if closes[i] > closes[i - 1])
    consistency = up_days / (len(closes) - 1)
    # Combine: 60 % weight on direction, 40 % on consistency
    raw = (ret_score * 0.6 + (consistency * 2 - 1) * 0.4)   # range −1 → +1
    normalised = (raw + 1) / 2                               # 0 → 1
    return int(round(_clamp(normalised * max_pts, 0, max_pts)))


def _sr_quality_score(closes: list[float], current_price: float, max_pts: int = 25) -> int:
    """Score support/resistance quality at current_price.

    Looks for price clusters (≥3 closes within 1.5 % of a level) in the
    historical window.  A signal near a tested support level scores higher.
    Scoring:
      - Touches of support within 3 % of current price → high score
      - Proximity to a resistance cluster → moderate score (breakout context)
      - No clear level nearby → low score
    """
    if len(closes) < 10 or current_price <= 0:
        return 12   # neutral

    band = 0.015  # 1.5 % cluster band

    # Find levels with ≥ 3 touches
    levels: list[float] = []
    for ref in closes:
        touches = sum(1 for c in closes if abs(c - ref) / ref <= band)
        if touches >= 3 and not any(abs(ref - lv) / ref <= band for lv in levels):
            levels.append(ref)

    if not levels:
        return 12

    # Classify levels relative to current price
    supports    = [lv for lv in levels if lv <= current_price * 1.02]
    resistances = [lv for lv in levels if lv >  current_price * 1.02]

    support_proximity  = 0.0
    resistance_context = 0.0

    if supports:
        nearest_sup = max(supports)
        dist = abs(current_price - nearest_sup) / current_price
        # Within 5 % of support = ideal
        support_proximity = _clamp(1.0 - dist / 0.05, 0.0, 1.0)

    if resistances:
        nearest_res = min(resistances)
        dist = (nearest_res - current_price) / current_price
        # Just broke above resistance (0-2 % above) = bonus
        resistance_context = _clamp(1.0 - dist / 0.10, 0.0, 1.0) * 0.5

    raw = support_proximity * 0.7 + resistance_context
    return int(round(_clamp(raw * max_pts, 0, max_pts)))


def _hist_vol_score(closes: list[float], max_pts: int = 25) -> int:
    """Score based on historical volatility regime.

    Low-to-moderate volatility (1-3 % daily avg move) scores best.
    Very low (< 0.5 %, dead coin) and very high (> 6 %) score poorly.
    """
    if len(closes) < 5:
        return 12   # neutral

    avg_daily_move = average([abs(percent_change(closes[i-1], closes[i]))
                              for i in range(1, len(closes))])
    # Score peaks at 1.5 % daily move, falls off on both sides
    ideal = 1.5
    deviation = abs(avg_daily_move - ideal) / ideal
    raw = _clamp(1.0 - deviation * 0.6, 0.0, 1.0)
    return int(round(_clamp(raw * max_pts, 0, max_pts)))


def historical_pattern_score(coin: str, current_price: float) -> HistoricalPatternScore:
    """Fetch daily candles and compute the five historical sub-scores.

    Tries USDT pair first, then INR, then BTC.  Returns all-neutral (50)
    score on failure so signals are never blocked by a candle fetch error.
    """
    candles: list[dict] = []
    for pair in _coin_to_pair(coin):
        candles = _fetch_daily_candles(pair, days=95)
        if candles:
            break

    if not candles:
        # No data → neutral score (50) so ranking still works
        return HistoricalPatternScore(12, 12, 12, 12, 12, 60)

    # Sort ascending by time
    candles = sorted(candles, key=lambda c: c.get("time", 0))
    closes  = [float(c.get("close", c.get("c", 0))) for c in candles if c.get("close", c.get("c", 0))]

    if len(closes) < 5:
        return HistoricalPatternScore(12, 12, 12, 12, 12, 60)

    trend_7d   = _trend_score_from_closes(closes[-7:]  if len(closes) >= 7  else closes)
    trend_30d  = _trend_score_from_closes(closes[-30:] if len(closes) >= 30 else closes)
    trend_90d  = _trend_score_from_closes(closes[-90:] if len(closes) >= 90 else closes)
    sr_quality = _sr_quality_score(closes, current_price)
    hist_vol   = _hist_vol_score(closes[-90:] if len(closes) >= 90 else closes)

    # Cap each sub-score to its max of 25, total to 100
    t7  = int(_clamp(trend_7d,   0, 25))
    t30 = int(_clamp(trend_30d,  0, 25))
    t90 = int(_clamp(trend_90d,  0, 25))
    sr  = int(_clamp(sr_quality, 0, 25))
    hv  = int(_clamp(hist_vol,   0, 25))
    total = int(_clamp(t7 + t30 + t90 + sr + hv, 0, 100))

    return HistoricalPatternScore(
        trend_7d=t7, trend_30d=t30, trend_90d=t90,
        sr_quality=sr, hist_vol=hv, total=total,
    )


# =============================================================================
# HISTORICAL BOOTSTRAP
# =============================================================================
# On startup, fetches 5-minute candles from CoinDCX to pre-fill price_history
# so EMA, MTF, and Phase 5 scoring work immediately without a 4-hour warm-up.
#
# Candle endpoint (public, no API key):
#   GET https://public.coindcx.com/market_data/candles/
#       ?pair=B-{COIN}_{QUOTE}&interval=5m&limit=120
#
# The result is mapped to the existing price_history format:
#   { "time": datetime, "price": close, "volume": volume }

BOOTSTRAP_CANDLES_URL = COINDCX_CANDLES_URL   # same base URL already defined
BOOTSTRAP_INTERVAL    = "5m"
BOOTSTRAP_LIMIT       = PRICE_HISTORY_LIMIT    # 120 candles = 10 h of 5-min data

# Minimum ticks needed for each readiness check
_READY_EMA    = EMA_SLOW_PERIOD   # 21
_READY_MTF_5M = MTF_5M_WINDOW     # 10
_READY_MTF_15 = MTF_15M_WINDOW    # 24
_READY_MTF_1H = MTF_1H_WINDOW     # 48
_READY_P5     = 20                # phase5_score asks for 20+ ticks


@dataclass
class BootstrapResult:
    coins_attempted: int = 0
    coins_loaded:    int = 0
    coins_failed:    int = 0
    avg_history_len: float = 0.0
    ema_ready:       bool = False
    mtf_ready:       bool = False
    phase5_ready:    bool = False
    duration_s:      float = 0.0
    failed_coins:    list = None   # type: ignore[assignment]

    def __post_init__(self):
        if self.failed_coins is None:
            self.failed_coins = []

    def summary_lines(self) -> list[str]:
        return [
            "[Bootstrap] Startup history pre-load complete",
            f"  Coins attempted : {self.coins_attempted}",
            f"  Successfully loaded : {self.coins_loaded}",
            f"  Failed          : {self.coins_failed}",
            f"  Avg history len : {self.avg_history_len:.1f} ticks",
            f"  Ready for EMA   : {'YES' if self.ema_ready else 'NO'}",
            f"  Ready for MTF   : {'YES' if self.mtf_ready else 'NO'}",
            f"  Ready for Phase5: {'YES' if self.phase5_ready else 'NO'}",
            f"  Duration        : {self.duration_s:.1f}s",
        ]


def get_historical_performance(coin: str) -> dict:
    """Fetch 7-day, 14-day, 30-day, and 90-day price change for *coin* from CoinDCX daily candles.

    Reuses _fetch_daily_candles() and _candle_cache — a single request of
    days=95 covers all four windows; if the coin's candles were already cached
    (e.g. by historical_pattern_score) no extra network call is made.

    Tries pairs in order: USDT → INR → BTC.
    Returns gracefully on any failure — never raises.

    Return dict:
        {
            "coin":      str,
            "perf_7d":   float | None,   # % change over last 7 daily closes
            "perf_14d":  float | None,   # % change over last 14 daily closes
            "perf_30d":  float | None,   # % change over last 30 daily closes
            "perf_90d":  float | None,   # % change over last 90 daily closes
            "source":    str | None,      # pair used, e.g. "B-BTC_USDT"
            "error":     str | None,      # set if fetch failed
        }
    """
    def _pct(closes: list, n: int):
        """Return n-day % change from the tail of closes, or None."""
        window = closes[-(n + 1):]   # need n+1 points for n changes
        if len(window) < 2:
            return None
        o = window[0]; c = window[-1]
        return round((c - o) / o * 100, 4) if o > 0 else None

    for pair in _coin_to_pair(coin):
        candles = _fetch_daily_candles(pair, days=95)   # 95 days covers 90d window
        if not candles:
            continue

        candles_sorted = sorted(candles, key=lambda c: c.get("time", 0))
        closes = [
            float(c.get("close", c.get("c", 0)) or 0)
            for c in candles_sorted
            if float(c.get("close", c.get("c", 0)) or 0) > 0
        ]

        if len(closes) < 2:
            continue

        return {
            "coin":     coin,
            "perf_7d":  _pct(closes, 7),
            "perf_14d": _pct(closes, 14),
            "perf_30d": _pct(closes, 30),
            "perf_90d": _pct(closes, 90),
            "source":   pair,
            "error":    None,
        }

    return {
        "coin":     coin,
        "perf_7d":  None,
        "perf_14d": None,
        "perf_30d": None,
        "perf_90d": None,
        "source":   None,
        "error":    "no candle data available",
    }



def _bootstrap_pair_candidates(coin: str) -> list[tuple[str, str]]:
    """Return (pair_string, quote) candidates in priority order."""
    coin = coin.upper()
    return [(f"B-{coin}_INR", "INR"), (f"B-{coin}_USDT", "USDT")]


def _fetch_bootstrap_candles(coin: str) -> list[dict]:
    """Fetch 5-min candles for *coin* synchronously.

    Tries INR pair first (higher liquidity on CoinDCX), then USDT.
    Returns [] on any failure — callers must degrade gracefully.
    """
    for pair, _quote in _bootstrap_pair_candidates(coin):
        try:
            resp = requests.get(
                BOOTSTRAP_CANDLES_URL,
                params={
                    "pair":     pair,
                    "interval": BOOTSTRAP_INTERVAL,
                    "limit":    BOOTSTRAP_LIMIT,
                },
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            if resp.status_code != 200:
                continue
            data = resp.json()
            if isinstance(data, list) and len(data) >= 2:
                return data
        except Exception:
            logger.debug("Bootstrap candle fetch failed: coin=%s pair=%s", coin, pair, exc_info=True)
    return []


def _candles_to_history(candles: list[dict]) -> list[dict]:
    """Convert raw CoinDCX candles to price_history format.

    CoinDCX candle keys: open, high, low, close, volume, time (ms epoch).
    Maps to: { time: datetime (UTC), price: close, volume: volume }
    """
    result = []
    for c in candles:
        try:
            close  = float(c.get("close",  c.get("c", 0)) or 0)
            volume = float(c.get("volume", c.get("v", 0)) or 0)
            ts_ms  = int(c.get("time",     c.get("t", 0)) or 0)
            if close <= 0 or ts_ms <= 0:
                continue
            result.append({
                "time":   datetime.fromtimestamp(ts_ms / 1000, tz=timezone.utc),
                "price":  close,
                "volume": volume,
            })
        except (TypeError, ValueError, KeyError):
            continue
    # Sort ascending by time so the newest tick is last (matches live append order)
    result.sort(key=lambda x: x["time"])
    return result[-PRICE_HISTORY_LIMIT:]   # respect the cap


async def bootstrap_price_history(
    coins: list[str],
    price_history: dict,
    concurrency: int = BOOTSTRAP_CONCURRENCY,
) -> BootstrapResult:
    """Pre-fill *price_history* for all *coins* from CoinDCX 5-min candles.

    Uses a semaphore to cap parallel HTTP requests.  Already-populated coins
    are skipped so the function is safe to call after partial progress.

    Args:
        coins:         List of coin symbols to bootstrap (e.g. ["BTC", "SOL"])
        price_history: The Scanner.price_history defaultdict to populate in-place
        concurrency:   Max parallel requests (default: BOOTSTRAP_CONCURRENCY)

    Returns:
        BootstrapResult with summary statistics and readiness flags.
    """
    import time as _time

    t_start  = _time.monotonic()
    sem      = asyncio.Semaphore(concurrency)
    results  = {}          # coin → list[dict] | None

    async def _fetch_one(coin: str) -> None:
        # Skip if already pre-loaded (e.g. from watchlist pass)
        if len(price_history.get(coin, [])) >= _READY_EMA:
            results[coin] = price_history[coin]
            return
        async with sem:
            candles = await asyncio.to_thread(_fetch_bootstrap_candles, coin)
        if candles:
            results[coin] = _candles_to_history(candles)
        else:
            results[coin] = None

    await asyncio.gather(*[_fetch_one(c) for c in coins], return_exceptions=False)

    # Write results into price_history
    loaded       = 0
    failed_coins = []
    hist_lens    = []

    for coin, history in results.items():
        if history:
            price_history[coin] = history
            loaded += 1
            hist_lens.append(len(history))
        else:
            failed_coins.append(coin)
            if failed_coins and len(failed_coins) <= 5:
                logger.warning("Bootstrap failed for coin=%s", coin)

    total      = len(coins)
    failed     = total - loaded
    avg_len    = sum(hist_lens) / len(hist_lens) if hist_lens else 0.0
    min_len    = min(hist_lens) if hist_lens else 0

    t_end = _time.monotonic()
    result = BootstrapResult(
        coins_attempted = total,
        coins_loaded    = loaded,
        coins_failed    = failed,
        avg_history_len = avg_len,
        ema_ready       = min_len >= _READY_EMA,
        mtf_ready       = min_len >= _READY_MTF_1H,   # 1h is the hardest bar
        phase5_ready    = min_len >= _READY_P5,
        duration_s      = t_end - t_start,
        failed_coins    = failed_coins,
    )

    for line in result.summary_lines():
        logger.info(line)
    print("\n".join(result.summary_lines()))

    return result

# =============================================================================
# PHASE 5 QUALITY SCORING
# =============================================================================
# Each dimension is scored 0-25.  Final = sum of the four = 0-100.
# The score is computed from existing price_history — no new API calls.
#
# Dimensions:
#   Trend Quality   – How cleanly and consistently the trend is moving up
#   Pullback Quality– Whether price pulled back healthily before resuming
#   Momentum        – Rate and acceleration of the move
#   Risk/Reward     – Estimated upside vs recent swing low (reward/risk ratio)

@dataclass(frozen=True)
class Phase5Score:
    trend_quality: int        # 0-25
    pullback_quality: int     # 0-25
    momentum: int             # 0-25
    risk_reward: int          # 0-25
    total: int                # 0-100

    def as_dict(self) -> dict:
        return {
            "trend_quality":    self.trend_quality,
            "pullback_quality": self.pullback_quality,
            "momentum":         self.momentum,
            "risk_reward":      self.risk_reward,
            "total":            self.total,
        }


def _clamp(value: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, value))


def phase5_score(history: list[dict]) -> Phase5Score:
    """Compute the four Phase 5 quality dimensions from price history.

    Requires at least 6 data points; returns all-zero score otherwise.
    All maths uses data already in price_history — no extra network calls.
    """
    prices = [item["price"] for item in history]
    volumes = [item["volume"] for item in history]

    if len(prices) < 6:
        return Phase5Score(0, 0, 0, 0, 0)

    # ── Trend Quality (0-25) ─────────────────────────────────────────────────
    # Measures how consistently prices are making higher-highs / higher-lows
    # over the last 20 bars, scaled by EMA separation.
    window = prices[-20:] if len(prices) >= 20 else prices
    up_moves = sum(1 for i in range(1, len(window)) if window[i] > window[i - 1])
    consistency = up_moves / (len(window) - 1) if len(window) > 1 else 0.0

    fast_e = ema(prices, EMA_FAST_PERIOD)
    slow_e = ema(prices, EMA_SLOW_PERIOD)
    ema_sep = (fast_e[-1] - slow_e[-1]) / slow_e[-1] * 100 if slow_e[-1] else 0.0
    ema_sep_score = _clamp(ema_sep / 2.0, 0.0, 1.0)   # 2 % separation → full marks

    tq_raw = (consistency * 0.6 + ema_sep_score * 0.4) * 25
    trend_quality = int(round(_clamp(tq_raw, 0, 25)))

    # ── Pullback Quality (0-25) ──────────────────────────────────────────────
    # A healthy pullback retraces 20-50 % of the prior leg, then resumes.
    # Score is highest when retracement is in the "golden zone" (30-50 %).
    # We look at the last 10 bars to detect a swing high → dip → recovery.
    pb_window = prices[-10:] if len(prices) >= 10 else prices
    if len(pb_window) >= 4:
        swing_high = max(pb_window[:-2])
        swing_low  = min(pb_window[1:-1])
        current_p  = pb_window[-1]
        prior_base = pb_window[0]

        leg_size = swing_high - prior_base
        if leg_size > 0 and swing_high > swing_low:
            retracement = (swing_high - swing_low) / leg_size
            # Peak score at 38 % retracement, falls off outside 20-60 %
            ideal = 0.38
            deviation = abs(retracement - ideal) / ideal
            pb_raw = _clamp(1.0 - deviation, 0.0, 1.0)
            # Require price to have recovered from the dip
            recovered = 1.0 if current_p > swing_low + (swing_high - swing_low) * 0.5 else 0.4
            pullback_quality = int(round(_clamp(pb_raw * recovered * 25, 0, 25)))
        else:
            pullback_quality = 0
    else:
        pullback_quality = 0

    # ── Momentum Score (0-25) ────────────────────────────────────────────────
    # Blends three signals:
    #   1. Recent price rate-of-change (last 3 bars)
    #   2. Volume acceleration (current vs 10-bar average)
    #   3. Whether momentum is accelerating (last move > previous move)
    roc_3 = percent_change(prices[-4], prices[-1]) if len(prices) >= 4 else 0.0
    roc_score = _clamp(roc_3 / 6.0, 0.0, 1.0)   # 6 % move in 3 bars = full marks

    recent_vol = average(volumes[-3:]) if len(volumes) >= 3 else volumes[-1]
    base_vol   = average(volumes[-13:-3]) if len(volumes) >= 13 else average(volumes)
    vol_acc    = _clamp((recent_vol / base_vol - 1.0) / 2.0, 0.0, 1.0) if base_vol else 0.0

    acc = 0.0
    if len(prices) >= 3:
        move_now  = prices[-1] - prices[-2]
        move_prev = prices[-2] - prices[-3]
        if move_prev != 0:
            acc = _clamp(move_now / abs(move_prev) - 1.0, 0.0, 1.0)

    mom_raw = (roc_score * 0.5 + vol_acc * 0.3 + acc * 0.2) * 25
    momentum = int(round(_clamp(mom_raw, 0, 25)))

    # ── Risk/Reward Score (0-25) ─────────────────────────────────────────────
    # Estimates reward/risk using:
    #   Risk   = distance from current price to recent swing low (10-bar)
    #   Reward = distance from current price to recent swing high projected
    #            by the same magnitude as the largest prior up-leg
    rr_window = prices[-15:] if len(prices) >= 15 else prices
    recent_low  = min(rr_window)
    recent_high = max(rr_window)
    cur = prices[-1]

    risk   = cur - recent_low  if cur > recent_low  else 0.0
    reward = recent_high - cur if recent_high > cur else (cur * 0.03)   # assume 3 % if at high

    if risk > 0:
        rr_ratio = reward / risk
        # Score peaks at RR ≥ 3:1
        rr_raw = _clamp(rr_ratio / 3.0, 0.0, 1.0) * 25
    else:
        rr_raw = 12.5   # neutral when no risk can be measured

    risk_reward = int(round(_clamp(rr_raw, 0, 25)))

    total = trend_quality + pullback_quality + momentum + risk_reward
    return Phase5Score(
        trend_quality=trend_quality,
        pullback_quality=pullback_quality,
        momentum=momentum,
        risk_reward=risk_reward,
        total=total,
    )


def ema(values: list[float], period: int) -> list[float]:
    if not values:
        return []

    multiplier = 2 / (period + 1)
    result = [values[0]]
    for value in values[1:]:
        result.append((value - result[-1]) * multiplier + result[-1])
    return result


def percent_change(old: float, new: float) -> float:
    if old == 0:
        return 0.0
    return ((new - old) / old) * 100


def average(values: list[float]) -> float:
    return sum(values) / len(values) if values else 0.0


def volatility(prices: list[float]) -> float:
    moves = [abs(percent_change(prices[i - 1], prices[i])) for i in range(1, len(prices))]
    return average(moves)


def trend_summary(history: list[dict]) -> dict:
    if len(history) < 2:
        return {"trend": "warming up", "move_percent": 0.0}

    prices = [item["price"] for item in history]
    lookback = min(10, len(prices) - 1)
    move = percent_change(prices[-lookback - 1], prices[-1])
    fast = ema(prices, EMA_FAST_PERIOD)[-1]
    slow = ema(prices, EMA_SLOW_PERIOD)[-1]

    if fast > slow and move > 0:
        trend = "uptrend"
    elif fast < slow and move < 0:
        trend = "downtrend"
    else:
        trend = "sideways"

    return {
        "trend": trend,
        "move_percent": move,
        "ema_fast": fast,
        "ema_slow": slow,
    }



# =============================================================================
# MULTI-TIMEFRAME ANALYSIS
# =============================================================================
# The scanner stores every tick in price_history (up to PRICE_HISTORY_LIMIT).
# Scan interval is 300 s (5 min).  So:
#   5 m  ≈  last 1  tick  — EMA crossover on the freshest sample window
#   15 m ≈  last 3  ticks — trend over ~3 intervals
#   1 h  ≈  last 12 ticks — trend over ~12 intervals
#
# "Bullish" for each frame means fast EMA > slow EMA AND price is rising.



def _frame_bullish(all_prices: list[float], window: int) -> bool:
    """Return True when the *window*-tick slice looks bullish.

    Computes EMAs on the FULL price history so the periods are never
    artificially collapsed by a short window.  Then checks that:
      1. fast EMA > slow EMA  (trend direction on full context)
      2. price rose over the window slice  (local momentum)
    Falls back to momentum-only when fewer than EMA_SLOW_PERIOD ticks
    are available (cold-start grace period).
    """
    if len(all_prices) < 2:
        return False
    slice_prices = all_prices[-window:] if len(all_prices) >= window else all_prices
    if len(slice_prices) < 2:
        return False
    # Use full history for EMA so periods stay at 9 / 21
    if len(all_prices) >= EMA_SLOW_PERIOD:
        fast_vals = ema(all_prices, EMA_FAST_PERIOD)
        slow_vals = ema(all_prices, EMA_SLOW_PERIOD)
        ema_bullish = fast_vals[-1] > slow_vals[-1]
    else:
        # Not enough history for reliable EMA — use momentum only
        ema_bullish = all_prices[-1] > all_prices[0]
    # Local momentum: price rose over the window
    momentum_bullish = slice_prices[-1] > slice_prices[0]
    return ema_bullish and momentum_bullish


# MTF diagnostic counters — track how many coins reach each alignment level
# Keyed by alignment label; reset each process restart.
_mtf_counts: dict[str, int] = {
    "5m_only":   0,   # 5m bull, 15m not bull
    "15m_only":  0,   # 15m bull, 5m not bull
    "5m_15m":    0,   # 5m + 15m bull, 1h not bull
    "5m_15m_1h": 0,   # all three bull
    "none":      0,   # none bullish → rejected
}

# Diagnostic counters shown by /mtf — reset each restart
_mtf_debug: dict[str, int] = {}

# First 10 MTF failures logged for /mtf inspection
_mtf_failures: list[dict] = []


def multi_timeframe_check(history: list[dict]) -> dict:
    """Evaluate bullish alignment across 5 m, 15 m, and 1 h frames.

    Relaxed rules (v5):
        CANDIDATE     – 5m OR 15m bullish
        STRONG SIGNAL – 5m AND 15m bullish
        PREMIUM       – 5m AND 15m AND 1h bullish

    Returns a dict with keys:
        tf_5m_bull   (bool) – 5-minute frame is bullish
        tf_15m_bull  (bool) – 15-minute frame is bullish
        tf_1h_bull   (bool) – 1-hour frame is bullish
        candidate_ok (bool) – passes minimum bar (5m OR 15m)
        strong_ok    (bool) – 5m AND 15m bullish
        premium_ok   (bool) – 5m AND 15m AND 1h bullish
        alignment    (str)  – human-readable label for diagnostics
    """
    prices = [item["price"] for item in history]

    # Pass full price list so EMA periods stay at 9/21 regardless of window size
    tf_5m  = _frame_bullish(prices, MTF_5M_WINDOW)
    tf_15m = _frame_bullish(prices, MTF_15M_WINDOW)
    tf_1h  = _frame_bullish(prices, MTF_1H_WINDOW)

    # Debug counters for /mtf command
    _mtf_debug["coins_checked"] = _mtf_debug.get("coins_checked", 0) + 1
    if len(prices) < MTF_5M_WINDOW:
        _mtf_debug["insufficient_history"] = _mtf_debug.get("insufficient_history", 0) + 1
    if tf_5m:
        _mtf_debug["5m_bullish"] = _mtf_debug.get("5m_bullish", 0) + 1
    if tf_15m:
        _mtf_debug["15m_bullish"] = _mtf_debug.get("15m_bullish", 0) + 1
    if tf_1h:
        _mtf_debug["1h_bullish"] = _mtf_debug.get("1h_bullish", 0) + 1
    if tf_5m and tf_15m and tf_1h:
        _mtf_debug["full_alignment"] = _mtf_debug.get("full_alignment", 0) + 1

    if tf_5m and tf_15m and tf_1h:
        alignment = "5m_15m_1h"
    elif tf_5m and tf_15m:
        alignment = "5m_15m"
    elif tf_5m and not tf_15m:
        alignment = "5m_only"
    elif tf_15m and not tf_5m:
        alignment = "15m_only"
    else:
        alignment = "none"

    _mtf_counts[alignment] = _mtf_counts.get(alignment, 0) + 1

    # Log first 10 "none" failures with detail for /mtf debugging
    if alignment == "none" and len(_mtf_failures) < 10:
        _mtf_failures.append({
            "history_len": len(prices),
            "tf_5m":  tf_5m,
            "tf_15m": tf_15m,
            "tf_1h":  tf_1h,
            "last_price": prices[-1] if prices else None,
            "first_price": prices[0] if prices else None,
        })

    return {
        "tf_5m_bull":   tf_5m,
        "tf_15m_bull":  tf_15m,
        "tf_1h_bull":   tf_1h,
        "candidate_ok": tf_5m or tf_15m,        # relaxed: OR instead of AND
        "strong_ok":    tf_5m and tf_15m,        # 5m AND 15m
        "premium_ok":   tf_5m and tf_15m and tf_1h,
        "alignment":    alignment,
    }


# =============================================================================
# MARKET STATE ENGINE
# =============================================================================
# Classifies the current market condition for a coin using ONLY data already
# computed by the scanner: EMA trend, price momentum, volume, and price
# structure (higher-highs / higher-lows / range).
#
# States: bull_trend | pullback | recovery | breakout | sideways | downtrend
#
# Called once per signal inside analyze_coin() — no extra API calls.

def detect_market_state(history: list[dict]) -> str:
    """Detect the current market state from price_history.

    Uses only data already available in the scanner:
      - EMA fast/slow relationship (full history)
      - Short-term momentum (last 3 bars)
      - Volume vs recent baseline
      - Higher-highs / higher-lows / lower-highs / lower-lows structure
      - Price position relative to recent range

    Returns one of:
        "bull_trend"  – EMA bullish, HH+HL structure, positive momentum
        "pullback"    – Bull context but price retreated from recent high
        "recovery"    – Previously weak, EMA reclaim + improving momentum
        "breakout"    – New local high + strong volume + strong momentum
        "sideways"    – No clear direction, flat range, weak momentum
        "downtrend"   – EMA bearish, LL+LH structure, negative momentum
    """
    if len(history) < 6:
        return "sideways"   # insufficient data → neutral default

    prices  = [item["price"]  for item in history]
    volumes = [item["volume"] for item in history]

    # ── EMA trend ────────────────────────────────────────────────────────────
    ema_bull = False
    if len(prices) >= EMA_SLOW_PERIOD:
        fast_e = ema(prices, EMA_FAST_PERIOD)
        slow_e = ema(prices, EMA_SLOW_PERIOD)
        ema_bull = fast_e[-1] > slow_e[-1]
        ema_separation = (fast_e[-1] - slow_e[-1]) / slow_e[-1] * 100 if slow_e[-1] else 0.0
    else:
        ema_bull = prices[-1] > prices[0]
        ema_separation = 0.0

    # ── Short-term momentum ───────────────────────────────────────────────────
    # Rate of change over last 3 bars
    momentum_3 = percent_change(prices[-4], prices[-1]) if len(prices) >= 4 else                  percent_change(prices[0],  prices[-1])
    momentum_positive = momentum_3 > 0.3   # > 0.3 % move = directional
    momentum_negative = momentum_3 < -0.3

    # ── Volume vs baseline ────────────────────────────────────────────────────
    recent_vol  = average(volumes[-3:]) if len(volumes) >= 3 else volumes[-1]
    baseline_vol = average(volumes[-13:-3]) if len(volumes) >= 13 else average(volumes)
    vol_ratio    = recent_vol / baseline_vol if baseline_vol else 1.0
    vol_spike    = vol_ratio > VOLUME_SPIKE_MULTIPLIER

    # ── Price structure (last 12 bars) ────────────────────────────────────────
    window = prices[-12:] if len(prices) >= 12 else prices
    n = len(window)

    # Split into halves and compare swing highs/lows
    first_half  = window[:n // 2]
    second_half = window[n // 2:]

    prev_high = max(first_half)
    prev_low  = min(first_half)
    curr_high = max(second_half)
    curr_low  = min(second_half)

    higher_highs = curr_high > prev_high
    higher_lows  = curr_low  > prev_low
    lower_highs  = curr_high < prev_high
    lower_lows   = curr_low  < prev_low

    # Price position in current range
    range_size   = curr_high - curr_low
    pos_in_range = (prices[-1] - curr_low) / range_size if range_size > 0 else 0.5
    near_top     = pos_in_range > 0.75
    near_bottom  = pos_in_range < 0.25

    # ── New local high detection ──────────────────────────────────────────────
    lookback = prices[-20:] if len(prices) >= 20 else prices
    new_local_high = prices[-1] >= max(lookback)

    # ── Classification (ordered from most specific to least) ─────────────────

    # BREAKOUT: new local high + volume surge + strong positive momentum
    if new_local_high and vol_spike and momentum_positive:
        return "breakout"

    # BULL TREND: EMA bullish + higher-highs + higher-lows + positive momentum
    if ema_bull and higher_highs and higher_lows and momentum_positive:
        return "bull_trend"

    # PULLBACK: bull EMA context but price retreated from recent high
    if ema_bull and near_bottom and not momentum_positive:
        return "pullback"

    # RECOVERY: previously weak (not full bull structure) + EMA reclaim +
    # momentum turning positive from low area
    if ema_bull and not (higher_highs and higher_lows) and momentum_positive:
        return "recovery"

    # DOWNTREND: EMA bearish + lower-highs + lower-lows + negative momentum
    if not ema_bull and lower_highs and lower_lows and momentum_negative:
        return "downtrend"

    # SIDEWAYS: no clear directional structure
    return "sideways"


# =============================================================================
# OPPORTUNITY TYPE ENGINE
# =============================================================================
# Converts market state + context signals into an actionable opportunity type
# that trading bots can consume directly.
#
# Inputs (all already computed before this is called in analyze_coin):
#   market_state   – from detect_market_state()
#   coin_class     – "A" | "B" | "C"
#   phase5_total   – Phase 5 quality score 0-100
#   mtf_alignment  – "5m_only" | "15m_only" | "5m_15m" | "5m_15m_1h" | "none"
#
# Base mapping (market_state → opportunity_type):
#   bull_trend → continuation
#   pullback   → accumulation
#   recovery   → recovery_trade
#   breakout   → momentum_trade
#   sideways   → watchlist
#   downtrend  → avoid        (always, regardless of other signals)
#
# Confidence is a 0-100 score stored alongside the type so future bots can
# filter by minimum confidence without needing to re-derive it.
# It is NOT used for signal filtering here — classification only.

_OPP_BASE: dict[str, str] = {
    "bull_trend": "continuation",
    "pullback":   "accumulation",
    "recovery":   "recovery_trade",
    "breakout":   "momentum_trade",
    "sideways":   "watchlist",
    "downtrend":  "avoid",
}

_OPP_LABELS: list[str] = [
    "accumulation", "recovery_trade", "momentum_trade",
    "continuation", "watchlist", "avoid",
]


# Priority thresholds (opportunity_score → priority label)
PRIORITY_LEVELS: list[tuple[int, str]] = [
    (90, "Elite"),
    (80, "High"),
    (70, "Medium"),
    (60, "Watch"),
    (0,  "Ignore"),
]

_CLASS_BONUS: dict[str, int] = {"A": 20, "B": 10, "C": 0}

_OPP_TYPE_BONUS: dict[str, int] = {
    "continuation":   15,
    "recovery_trade": 12,
    "accumulation":   10,
    "momentum_trade":  8,
    "watchlist":       0,
    "avoid":         -50,
}

_MTF_BONUS: dict[str, int] = {
    "5m_15m_1h": 20,
    "5m_15m":    10,
    "5m_only":    5,
    "15m_only":   5,
    "none":       0,
}
def calculate_risk_level(
    coin_class:       str,
    opportunity_type: str,
    mtf_alignment:    str,
    confidence:       int,
) -> str:
    """Classify risk as \"low\", \"medium\", or \"high\".

    Base risk is determined by coin class, opportunity type, and MTF alignment.
    Low confidence (<60) escalates risk by one level.

    Base rules (first match wins):
        avoid                         → high
        A + full MTF (5m+15m+1h)      → low
        A + 5m+15m                    → low
        A + single/no TF              → medium
        B (any MTF)                   → medium
        C + momentum_trade            → high
        C (other)                     → medium

    Confidence escalation:
        confidence < 60:  low→medium, medium→high, high→high
    """
    if opportunity_type == "avoid":
        return "high"

    full_mtf   = mtf_alignment == "5m_15m_1h"
    strong_mtf = mtf_alignment == "5m_15m"

    if coin_class == "A" and (full_mtf or strong_mtf):
        base = "low"
    elif coin_class == "A":
        base = "medium"
    elif coin_class == "B":
        base = "medium"
    elif opportunity_type == "momentum_trade":
        base = "high"
    else:
        base = "medium"

    if confidence < 60:
        if base == "low":
            return "medium"
        return "high"

    return base


def priority_from_score(opp_score: int) -> str:
    """Map an opportunity_score (0-100) to a priority label."""
    for threshold, label in PRIORITY_LEVELS:
        if opp_score >= threshold:
            return label
    return "Ignore"


def calculate_opportunity_score(
    coin_class:       str,
    opportunity_type: str,
    confidence:       int,
    mtf_alignment:    str,
    historical_score: int,
) -> int:
    """Rank-score a signal from 0-100 using classification + context signals.

    Components (additive, result capped at 100):
        Base                           30
        Coin class bonus (A/B/C)     0-20
        Opportunity type bonus       -50 to +15
        MTF alignment bonus           0-20
        Confidence contribution       0-10  (confidence/10)
        Historical score contribution 0-10  (hist/10, capped at 10)

    "avoid" signals always score 0 so they never surface.
    """
    if opportunity_type == "avoid":
        return 0

    score = 30
    score += _CLASS_BONUS.get(coin_class, 0)
    score += _OPP_TYPE_BONUS.get(opportunity_type, 0)
    score += _MTF_BONUS.get(mtf_alignment, 0)
    score += min(confidence // 10, 10)          # 0-10 from confidence
    score += min(historical_score // 10, 10)    # 0-10 from historical score

    return max(0, min(100, score))



def detect_opportunity_type(
    market_state: str,
    coin_class: str,
    phase5_total: int,
    mtf_alignment: str,
) -> tuple[str, int]:
    """Map market state + context to an opportunity type and confidence score.

    Returns:
        (opportunity_type: str, confidence: int 0-100)

    Downtrend always returns ("avoid", 0) regardless of other inputs.
    Classification only — does not affect signal generation or scoring.
    """
    # Hard rule: downtrend is always avoid
    if market_state == "downtrend":
        return ("avoid", 0)

    opp_type = _OPP_BASE.get(market_state, "watchlist")

    # ── Confidence scoring (additive, capped at 100) ─────────────────────────
    confidence = 40   # base

    # Coin class bonus
    if coin_class == "A":
        confidence += 25
    elif coin_class == "B":
        confidence += 15
    # class C: +0

    # MTF alignment bonus
    if mtf_alignment == "5m_15m_1h":
        confidence += 20
    elif mtf_alignment == "5m_15m":
        confidence += 12
    elif mtf_alignment in ("5m_only", "15m_only"):
        confidence += 5
    # "none": +0  (shouldn't reach here, but defensive)

    # Phase 5 quality bonus
    if phase5_total >= 75:
        confidence += 15
    elif phase5_total >= 50:
        confidence += 8
    elif phase5_total >= 25:
        confidence += 3

    confidence = min(confidence, 100)
    return (opp_type, confidence)


def smart_filter(signal: "Signal") -> bool:
    """Post-scoring gate: block weak/invalid signals before alert dispatch.

    Runs AFTER all scoring is complete — does not affect scores or ranking.
    Updates _filter_counts["smart_filter"] for /stats reporting.

    Returns True  → allow the signal through to alerts.
    Returns False → reject silently, increment counter.

    Rejection rules (any one is sufficient):
        1. priority == "Ignore"
        2. opportunity_type == "avoid"
        3. risk_level == "high" AND opp_confidence < 60
    """
    reject = (
        signal.priority.lower() == "ignore"
        or signal.opportunity_type == "avoid"
        or (signal.risk_level.lower() == "high" and signal.opp_confidence < 60)
    )
    if reject:
        _filter_counts["smart_filter"] = _filter_counts.get("smart_filter", 0) + 1
        return False
    return True



# Cache of avoid/recommended setup keys from learning_recommendations().
# Reset each restart; populated lazily on first call to learning_filter().
_learning_avoid_keys:       set | None = None
_learning_recommend_keys:   set | None = None
_learning_cache_updated_at: float = 0.0
_LEARNING_CACHE_TTL = 3600.0   # refresh once per hour


def _build_learning_key(signal: "Signal") -> str:
    """Composite key matching the format used by _setup_performance():
    "coin_class|market_state|opportunity_type|priority"
    """
    return (
        signal.coin_class + "|"
        + signal.market_state + "|"
        + signal.opportunity_type + "|"
        + signal.priority
    )


def _refresh_learning_cache(tracker) -> None:
    """Rebuild avoid/recommended key sets from the tracker's learning data."""
    global _learning_avoid_keys, _learning_recommend_keys, _learning_cache_updated_at
    import time as _time

    now = _time.monotonic()
    if (
        _learning_avoid_keys is not None
        and now - _learning_cache_updated_at < _LEARNING_CACHE_TTL
    ):
        return   # still fresh

    recs = tracker.learning_recommendations()

    def _key_from_desc(desc: str) -> str:
        """Extract the setup key from a recommendation description string.

        Format: "Prioritize A-class bull trend (continuation)  WR:...  [N eval]"
        We reverse-engineer the key by matching class, market_state, opp_type
        against the description text.  This is intentionally simple — the key
        is only used as a lookup, not for display.
        """
        # The _describe() format is:
        #   "Prioritize {cls}-class {ms} ({ot})  WR:..."
        # We parse it back out.
        try:
            # Strip action prefix
            parts = desc.split("-class ", 1)
            cls = parts[0].split()[-1].strip()
            rest = parts[1]   # "bull trend (continuation)  WR:..."
            # Find the opportunity type in parentheses
            ot_start = rest.find("(")
            ot_end   = rest.find(")")
            ot_raw   = rest[ot_start + 1:ot_end].replace(" ", "_")
            ms_raw   = rest[:ot_start].strip().replace(" ", "_")
            # Priority is not embedded in the description; we use "*" as wildcard
            return cls + "|" + ms_raw + "|" + ot_raw + "|*"
        except Exception:
            return ""

    _learning_avoid_keys       = {_key_from_desc(d) for d in recs.get("avoid", [])}
    _learning_recommend_keys   = {_key_from_desc(d) for d in recs.get("recommended", [])}
    _learning_cache_updated_at = now


def _matches_learning_key(signal_key: str, key_set: set) -> bool:
    """True if signal_key matches any entry in key_set.

    Keys use "*" as a wildcard for the priority segment since learning
    recommendations don't embed the exact priority level.
    """
    if signal_key in key_set:
        return True
    # Try wildcard match on last segment
    base = "|".join(signal_key.split("|")[:3]) + "|*"
    return base in key_set


def learning_filter(signal: "Signal", tracker) -> bool:
    """Post-smart_filter gate using the Learning Layer.

    Reads learning_recommendations() to build avoid/recommend key sets
    (cached for 1 hour to avoid repeated computation).

    Returns False (reject) if:
        - The signal's setup key matches an avoid entry AND does NOT
          match a recommended entry.

    Returns True (allow) in all other cases — including when there is
    insufficient data for recommendations (has_data=False), which ensures
    the filter is a no-op until enough history exists.

    Updates _filter_counts["learning_filter"] on rejection.
    """
    _refresh_learning_cache(tracker)

    # No-op when learning data isn't available yet
    if not _learning_avoid_keys:
        return True

    key = _build_learning_key(signal)
    if not key:
        return True

    in_avoid      = _matches_learning_key(key, _learning_avoid_keys)
    in_recommend  = _matches_learning_key(key, _learning_recommend_keys or set())

    if in_avoid and not in_recommend:
        _filter_counts["learning_filter"] = _filter_counts.get("learning_filter", 0) + 1
        return False

    return True



# Thresholds for historical_filter
HIST_FILTER_REJECT_BELOW  = -50.0   # perf_90d < -50% → reject
HIST_FILTER_FLAG_ABOVE    = 100.0   # perf_90d > +100% → flag (allow but tag reason)


def historical_filter(signal: "Signal") -> bool:
    """Post-learning_filter gate using exchange 90-day historical performance.

    Rules:
        Reject: exch_perf_90d < -50%  (coin has been in severe long-term decline)
        Flag:   exch_perf_90d > +100% (parabolic move — allow but log the flag)
        Allow:  all other cases, including when exch_perf_90d is None
                (coin has insufficient history → no-op, don't block)

    Updates _filter_counts["historical_filter"] on rejection.
    Does NOT affect scores, ranking, or any other signal field.
    """
    p90 = signal.exch_perf_90d

    if p90 is None:
        return True   # no data → allow (never block on missing data)

    if p90 < HIST_FILTER_REJECT_BELOW:
        _filter_counts["historical_filter"] = _filter_counts.get("historical_filter", 0) + 1
        logger.debug(
            "historical_filter rejected %s: exch_perf_90d=%.2f%%", signal.coin, p90
        )
        return False

    if p90 > HIST_FILTER_FLAG_ABOVE:
        logger.debug(
            "historical_filter flagged %s: exch_perf_90d=%.2f%% (parabolic)", signal.coin, p90
        )
        # Allow through — flag is informational only

    return True


def signal_tier(score: int) -> str:
    if score >= 85:
        return "PREMIUM"
    if score >= 70:
        return "STRONG SIGNAL"
    if score >= 60:
        return "CANDIDATE"
    return "IGNORE"


def format_price(price: float) -> str:
    text = f"{price:,.8f}".rstrip("0").rstrip(".")
    return text if text else "0"


def format_volume(volume: float) -> str:
    return f"{volume:,.2f}"


def format_percent(value: Optional[float]) -> str:
    if value is None:
        return "pending"
    return f"{value:+.2f}%"


# Tracks signals filtered at analysis stage (reset each scanner run)
_filter_counts: dict[str, int] = {"low_score": 0, "no_volume": 0, "no_ema": 0, "no_mtf": 0, "smart_filter": 0, "learning_filter": 0, "historical_filter": 0}


def analyze_coin(coin: str, history: list[dict]) -> list[Signal]:
    """Return qualifying signals for *coin*.

    Gates (in order):
    1. EMA crossover is MANDATORY
    2. Volume spike is MANDATORY
    3. MTF gate (relaxed v5): pass if 5m OR 15m is bullish
    4. Minimum score >= 60

    Tier assignment:
        5m only / 15m only  → CANDIDATE (capped)
        5m + 15m            → STRONG SIGNAL (capped, +5 score bonus)
        5m + 15m + 1h       → PREMIUM eligible (+10 score bonus)
    """
    if len(history) < 2:
        return []

    # Multi-timeframe pre-check — pass if 5m OR 15m is bullish
    mtf = multi_timeframe_check(history)
    if not mtf["candidate_ok"]:
        _filter_counts["no_mtf"] = _filter_counts.get("no_mtf", 0) + 1
        return []

    prices = [item["price"] for item in history]
    volumes = [item["volume"] for item in history]
    current_price = prices[-1]
    current_volume = volumes[-1]
    created_at = datetime.now(timezone.utc)
    score = 0
    reasons: list[str] = []

    fast = ema(prices, EMA_FAST_PERIOD)
    slow = ema(prices, EMA_SLOW_PERIOD)
    recent_move = percent_change(prices[-2], current_price)
    momentum_strength = max(recent_move, 0.0)

    previous_volumes = volumes[-(VOLUME_AVERAGE_PERIOD + 1):-1]
    average_volume = average(previous_volumes)
    volume_strength = current_volume / average_volume if average_volume else 0.0

    # --- Gate 1: EMA crossover is MANDATORY ---
    has_ema_crossover = False
    if len(fast) >= 2 and len(slow) >= 2:
        crossed_up = fast[-2] <= slow[-2] and fast[-1] > slow[-1]
        crossed_down = fast[-2] >= slow[-2] and fast[-1] < slow[-1]
        if crossed_up or crossed_down:
            has_ema_crossover = True
            score += 25
            reasons.append("EMA crossover")

        if fast[-1] > slow[-1] and recent_move > 0:
            score += 10
            reasons.append("Strong trend")

    if not has_ema_crossover:
        _filter_counts["no_ema"] += 1
        return []

    # --- Gate 2: Volume spike is MANDATORY ---
    has_volume_spike = volume_strength > VOLUME_SPIKE_MULTIPLIER
    if has_volume_spike:
        score += 20
        reasons.append("Volume spike")
    else:
        _filter_counts["no_volume"] += 1
        return []

    if recent_move >= MOMENTUM_THRESHOLD_PERCENT:
        score += 15
        reasons.append("Positive momentum")

    if len(prices) > VOLATILITY_LOOKBACK + 1:
        current_volatility = volatility(prices[-(VOLATILITY_LOOKBACK + 1):])
        baseline = volatility(prices[-(VOLATILITY_LOOKBACK * 2 + 1):-VOLATILITY_LOOKBACK])
        if baseline and current_volatility > baseline * VOLATILITY_SPIKE_MULTIPLIER:
            score += 10
            reasons.append("High volatility breakout")

    # --- Gate 3: Minimum score threshold raised to 60 ---
    if score < 60:
        _filter_counts["low_score"] += 1
        return []

    # MTF confirmation bonus / tier capping (v5 relaxed rules)
    #
    # Alignment   → tier cap          bonus
    # 5m only     → CANDIDATE max     +0
    # 15m only    → CANDIDATE max     +0
    # 5m + 15m    → STRONG SIGNAL max +5
    # 5m+15m+1h   → PREMIUM allowed   +10
    alignment = mtf["alignment"]

    if alignment == "5m_15m_1h":
        score = min(score + 10, 100)
        reasons.append("MTF: 5m+15m+1h aligned ⭐")
        effective_tier = signal_tier(score)          # can reach PREMIUM
    elif alignment == "5m_15m":
        score = min(score + 5, 100)
        reasons.append("MTF: 5m+15m aligned")
        # Cap at STRONG SIGNAL even if score would reach PREMIUM
        raw_tier = signal_tier(score)
        effective_tier = "STRONG SIGNAL" if raw_tier == "PREMIUM" else raw_tier
    elif alignment == "5m_only":
        reasons.append("MTF: 5m bullish only")
        effective_tier = "CANDIDATE"                 # cap at CANDIDATE
    else:  # 15m_only
        reasons.append("MTF: 15m bullish only")
        effective_tier = "CANDIDATE"                 # cap at CANDIDATE

    tier = effective_tier
    coin_class   = get_coin_class(coin)
    market_state = detect_market_state(history)
    # opportunity_type is computed after p5/hist so it has phase5_total;
    # we forward-declare here and fill below after scoring.
    # Phase 5 quality score (uses in-memory price history)
    p5 = phase5_score(history)

    # Historical Pattern Score (fetches daily candles, cached 1 h)
    hist = historical_pattern_score(coin, current_price)

    # Final Score = 40% scanner + 40% Phase5 + 20% historical
    scanner_norm = min(score, 100)
    final_score  = int(round(
        scanner_norm  * 0.40
        + p5.total    * 0.40
        + hist.total  * 0.20
    ))

    # Opportunity type — uses p5.total and mtf alignment
    opportunity_type, opp_confidence = detect_opportunity_type(
        market_state  = market_state,
        coin_class    = coin_class,
        phase5_total  = p5.total,
        mtf_alignment = mtf.get("alignment", "none"),
    )

    opportunity_score = calculate_opportunity_score(
        coin_class       = coin_class,
        opportunity_type = opportunity_type,
        confidence       = opp_confidence,
        mtf_alignment    = mtf.get("alignment", "none"),
        historical_score = hist.total,
    )
    priority   = priority_from_score(opportunity_score)
    risk_level = calculate_risk_level(
        coin_class       = coin_class,
        opportunity_type = opportunity_type,
        mtf_alignment    = mtf.get("alignment", "none"),
        confidence       = opp_confidence,
    )

    return [
        Signal(
            coin=coin,
            kind=tier.lower().replace(" ", "_"),
            score=score,
            message="; ".join(reasons),
            price=current_price,
            volume=current_volume,
            created_at=created_at,
            tier=tier,
            reasons=reasons,
            volume_strength=volume_strength,
            momentum_strength=momentum_strength,
            model_version=MODEL_VERSION,
            phase5_trend=p5.trend_quality,
            phase5_pullback=p5.pullback_quality,
            phase5_momentum=p5.momentum,
            phase5_risk_reward=p5.risk_reward,
            phase5_total=p5.total,
            final_score=final_score,
            hist_trend_7d=hist.trend_7d,
            hist_trend_30d=hist.trend_30d,
            hist_trend_90d=hist.trend_90d,
            hist_sr_quality=hist.sr_quality,
            hist_vol_score=hist.hist_vol,
            hist_total=hist.total,
            coin_class=coin_class,
            market_state=market_state,
            opportunity_type=opportunity_type,
            opp_confidence=opp_confidence,
            opportunity_score=opportunity_score,
            priority=priority,
            risk_level=risk_level,
        )
    ]


# =============================================================================
# PUBLIC MARKET DATA
# =============================================================================
# CoinDCX ticker is public, so no API key or secret key is needed.

class CoinDCXPublicClient:
    def fetch_tickers(self) -> list[dict]:
        response = requests.get(COINDCX_TICKER_URL, timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status()
        return response.json()

# =============================================================================
# SIGNAL PERFORMANCE TRACKING
# =============================================================================
# Signals are logged to JSON and evaluated later using public market prices.

class SignalPerformanceTracker:
    def __init__(self, path: str = SIGNAL_LOG_FILE, stats_path: str = STATS_FILE):
        self.path = Path(path)
        self.stats_path = Path(stats_path)
        self._data = self._load()

    def _load(self) -> dict:
        if not self.path.exists():
            return {"signals": []}

        try:
            data = json.loads(self.path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            logger.warning("Could not read signal performance file; starting fresh", exc_info=True)
            return {"signals": []}

        if not isinstance(data, dict):
            return {"signals": []}
        data.setdefault("signals", [])
        return data

    def save(self) -> None:
        write_json_safely(self.path, self._data)

    def save_stats(self) -> None:
        """Persist current performance stats snapshot to stats.json on Drive."""
        try:
            write_json_safely(self.stats_path, self.stats())
        except Exception:
            logger.warning("Could not save stats snapshot", exc_info=True)

    def log_signal(self, signal: Signal) -> None:
        if self._is_duplicate(signal):
            return

        self._data["signals"].append(
            {
                "id": f"{signal.coin}-{int(signal.created_at.timestamp())}-{signal.kind}",
                "timestamp": signal.created_at.isoformat(),
                "coin": signal.coin,
                "category": signal.tier.title(),
                "score": signal.score,
                "signal_price": signal.price,
                "reasons": signal.reasons,
                "model_version": signal.model_version,
                "phase5": {
                    "trend_quality":    signal.phase5_trend,
                    "pullback_quality": signal.phase5_pullback,
                    "momentum":         signal.phase5_momentum,
                    "risk_reward":      signal.phase5_risk_reward,
                    "total":            signal.phase5_total,
                },
                "final_score": signal.final_score,
                "coin_class":    signal.coin_class,
                "market_state":    signal.market_state,
                "opportunity_type": signal.opportunity_type,
                "opp_confidence":    signal.opp_confidence,
                "opportunity_score": signal.opportunity_score,
                "priority":          signal.priority,
                "risk_level":        signal.risk_level,
                "historical_score": {
                    "trend_7d":   signal.hist_trend_7d,
                    "trend_30d":  signal.hist_trend_30d,
                    "trend_90d":  signal.hist_trend_90d,
                    "sr_quality": signal.hist_sr_quality,
                    "hist_vol":   signal.hist_vol_score,
                    "total":      signal.hist_total,
                },
                "evaluations": {},
            }
        )
        self.save()

    def evaluate_due_signals(self, prices: dict[str, float]) -> int:
        now = datetime.now(timezone.utc)
        updated = 0

        for item in self._data["signals"]:
            signal_time = self._parse_time(item.get("timestamp"))
            if signal_time is None:
                continue

            coin = item.get("coin")
            current_price = prices.get(coin)
            if current_price is None or current_price <= 0:
                continue

            signal_price = self._to_float(item.get("signal_price"))
            if signal_price <= 0:
                continue

            evaluations = item.setdefault("evaluations", {})
            age = (now - signal_time).total_seconds()
            for label, seconds in EVALUATION_HORIZONS.items():
                if age >= seconds and label not in evaluations:
                    change = percent_change(signal_price, current_price)
                    evaluations[label] = {
                        "timestamp": now.isoformat(),
                        "price": current_price,
                        "change_percent": change,
                    }
                    updated += 1

        if updated:
            self.save()
            self.save_stats()
        return updated

    def stats(self) -> dict:
        signals = self._data["signals"]
        latest_changes = [self._latest_change(item) for item in signals]
        completed = [change for change in latest_changes if change is not None]
        winners = sum(1 for change in completed if change > 0)
        losers = sum(1 for change in completed if change <= 0)

        stats = {
            "last_updated": datetime.now(timezone.utc).isoformat(),
            "total_signals": len(signals),
            "winning_signals": winners,
            "losing_signals": losers,
            "win_rate": (winners / len(completed) * 100) if completed else 0.0,
            "average_returns": {
                label: self._average_return(label)
                for label in EVALUATION_HORIZONS
            },
            "score_groups": self._score_group_stats(),
            # Live filter counters (reset each process restart)
            "filtered_low_score": _filter_counts["low_score"],
            "filtered_no_volume": _filter_counts["no_volume"],
            "filtered_no_ema": _filter_counts["no_ema"],
            "filtered_no_mtf": _filter_counts.get("no_mtf", 0),
            "filtered_smart":    _filter_counts.get("smart_filter", 0),
            "filtered_learning":   _filter_counts.get("learning_filter", 0),
            "filtered_historical": _filter_counts.get("historical_filter", 0),
            # MTF alignment diagnostics (counts per level, reset each restart)
            "mtf_diagnostics": {
                "5m_only":   _mtf_counts.get("5m_only",   0),
                "15m_only":  _mtf_counts.get("15m_only",  0),
                "5m_15m":    _mtf_counts.get("5m_15m",    0),
                "5m_15m_1h": _mtf_counts.get("5m_15m_1h", 0),
                "none":      _mtf_counts.get("none",      0),
            },
            # Class analytics
            "class_distribution":        self._class_distribution(),
            "class_performance":         self._class_performance(),
            "market_state_distribution":   self._market_state_distribution(),
            "priority_distribution":       self._priority_distribution(),
            "priority_performance":        self._priority_performance(),
            "best_setups":                 self.best_setups(),
            "worst_setups":                self.worst_setups(),
            "learning_recommendations":    self.learning_recommendations(),
            "opportunity_distribution":    self._opportunity_distribution(),
            "opportunity_performance":     self._opportunity_performance(),
            # Per-version breakdown
            "model_stats": {MODEL_VERSION: self._model_stats(MODEL_VERSION)},
        }
        write_json_safely(self.stats_path, stats)
        return stats
    def top_ranked_signals(self, limit: int = 10) -> list[dict]:
        """Return up to *limit* stored signals sorted by ranking engine output.

        Sort order:
            1. opportunity_score  (desc)
            2. opp_confidence     (desc)
            3. timestamp          (desc — newest first as tiebreaker)

        Filters:
            - Signals missing opportunity_score are skipped (pre-v12 records)
            - Signals with priority == "Ignore" are skipped
        """
        ranked = []
        for item in self._data["signals"]:
            opp_sc = item.get("opportunity_score")
            if opp_sc is None:
                continue   # pre-v12 signal, no ranking data
            pri = item.get("priority", "Ignore")
            if pri.lower() == "ignore":
                continue
            ranked.append(item)

        ranked.sort(
            key=lambda s: (
                self._to_float(s.get("opportunity_score", 0)),
                self._to_float(s.get("opp_confidence", 0)),
                s.get("timestamp", ""),
            ),
            reverse=True,
        )
        return ranked[:limit]

    def recent_signals(self, limit: int = 10, current_prices: Optional[dict[str, float]] = None) -> list[dict]:
        current_prices = current_prices or {}
        items = sorted(
            self._data["signals"],
            key=lambda item: item.get("timestamp", ""),
            reverse=True,
        )[:limit]

        results = []
        for item in items:
            signal_price = self._to_float(item.get("signal_price"))
            current_price = current_prices.get(item.get("coin"))
            current_change = None
            if current_price and signal_price > 0:
                current_change = percent_change(signal_price, current_price)

            results.append(
                {
                    **item,
                    "current_change_percent": current_change,
                    "latest_change_percent": self._latest_change(item),
                }
            )
        return results

    def _average_return(self, label: str) -> Optional[float]:
        values = []
        for item in self._data["signals"]:
            evaluation = item.get("evaluations", {}).get(label)
            if evaluation is not None:
                values.append(self._to_float(evaluation.get("change_percent")))
        return average(values) if values else None

    def _model_stats(self, version: str) -> dict:
        """Compute win-rate and average returns for a single model version."""
        subset = [s for s in self._data["signals"] if s.get("model_version") == version]
        if not subset:
            return {"total": 0, "win_rate": 0.0, "average_returns": {k: None for k in EVALUATION_HORIZONS}}

        latest_changes = [self._latest_change(item) for item in subset]
        completed = [c for c in latest_changes if c is not None]
        winners = sum(1 for c in completed if c > 0)

        avg_returns = {}
        for label in EVALUATION_HORIZONS:
            vals = [
                self._to_float(item.get("evaluations", {}).get(label, {}).get("change_percent"))
                for item in subset
                if label in item.get("evaluations", {})
            ]
            avg_returns[label] = average(vals) if vals else None

        return {
            "total": len(subset),
            "evaluated": len(completed),
            "win_rate": (winners / len(completed) * 100) if completed else 0.0,
            "average_returns": avg_returns,
        }

    def _priority_distribution(self) -> dict:
        """Count signals per priority level across all logged signals."""
        labels = [label for _, label in PRIORITY_LEVELS]
        counts: dict[str, int] = {label: 0 for label in labels}
        for item in self._data["signals"]:
            pri = item.get("priority", "")
            if not pri:
                # Backfill for old signals that have opportunity_score stored
                opp_sc = item.get("opportunity_score")
                if opp_sc is not None:
                    pri = priority_from_score(int(opp_sc))
            if pri in counts:
                counts[pri] += 1
        return counts

    def _priority_performance(self) -> dict:
        """Compute win rate and avg returns per priority level.

        Same pattern as _opportunity_performance().
        Signals missing a priority field are skipped (pre-v12 records).
        """
        labels = [label for _, label in PRIORITY_LEVELS]
        buckets: dict[str, list] = {label: [] for label in labels}
        for item in self._data["signals"]:
            pri = item.get("priority", "")
            if not pri:
                continue   # no priority stored — skip
            buckets.setdefault(pri, []).append(item)

        result: dict[str, dict] = {}
        for pri, signals in buckets.items():
            if not signals:
                result[pri] = {
                    "total": 0, "evaluated": 0, "win_rate": 0.0,
                    "avg_returns": {h: None for h in EVALUATION_HORIZONS},
                }
                continue
            latest_changes = [self._latest_change(s) for s in signals]
            completed = [c for c in latest_changes if c is not None]
            winners   = sum(1 for c in completed if c > 0)
            avg_returns = {}
            for horizon in EVALUATION_HORIZONS:
                vals = [
                    self._to_float(
                        s.get("evaluations", {}).get(horizon, {}).get("change_percent")
                    )
                    for s in signals
                    if horizon in s.get("evaluations", {})
                ]
                avg_returns[horizon] = average(vals) if vals else None
            result[pri] = {
                "total":       len(signals),
                "evaluated":   len(completed),
                "win_rate":    (winners / len(completed) * 100) if completed else 0.0,
                "avg_returns": avg_returns,
            }
        return result

    def _setup_performance(self) -> dict:
        """Group signals by (coin_class, market_state, opportunity_type, priority)
        and compute win rate + avg returns per combination.

        Key format: "cls|market_state|opportunity_type|priority"
        Signals missing any of the four fields are skipped (pre-v12 records).
        """
        buckets: dict[str, list] = {}

        for item in self._data["signals"]:
            cls = item.get("coin_class", "")
            ms  = item.get("market_state", "")
            ot  = item.get("opportunity_type", "")
            pri = item.get("priority", "")
            if not (cls and ms and ot and pri):
                continue   # pre-v12 signal — skip
            key = cls + "|" + ms + "|" + ot + "|" + pri
            buckets.setdefault(key, []).append(item)

        result: dict[str, dict] = {}
        for key, signals in buckets.items():
            latest_changes = [self._latest_change(s) for s in signals]
            completed      = [c for c in latest_changes if c is not None]
            winners        = sum(1 for c in completed if c > 0)

            avg_returns: dict[str, object] = {}
            for horizon in EVALUATION_HORIZONS:
                vals = [
                    self._to_float(
                        s.get("evaluations", {}).get(horizon, {}).get("change_percent")
                    )
                    for s in signals
                    if horizon in s.get("evaluations", {})
                ]
                avg_returns[horizon] = average(vals) if vals else None

            result[key] = {
                "key":       key,
                "total":     len(signals),
                "evaluated": len(completed),
                "win_rate":  (winners / len(completed) * 100) if completed else 0.0,
                "avg_returns": avg_returns,
            }

        return result

    def best_setups(self, min_evaluated: int = 5, limit: int = 5) -> list:
        """Return top *limit* setups sorted by avg_24h desc, then win_rate desc.

        Only includes setups with at least min_evaluated completed evaluations.
        """
        perf = self._setup_performance()
        eligible = [
            v for v in perf.values()
            if v["evaluated"] >= min_evaluated
            and v["avg_returns"].get("24h") is not None
        ]
        eligible.sort(
            key=lambda v: (v["avg_returns"]["24h"], v["win_rate"]),
            reverse=True,
        )
        return eligible[:limit]

    def worst_setups(self, min_evaluated: int = 5, limit: int = 5) -> list:
        """Return bottom *limit* setups sorted by avg_24h asc, then win_rate asc.

        Only includes setups with at least min_evaluated completed evaluations.
        """
        perf = self._setup_performance()
        eligible = [
            v for v in perf.values()
            if v["evaluated"] >= min_evaluated
            and v["avg_returns"].get("24h") is not None
        ]
        eligible.sort(
            key=lambda v: (v["avg_returns"]["24h"], v["win_rate"]),
        )
        return eligible[:limit]


    def learning_recommendations(self) -> dict:
        """Derive actionable recommendations from historical setup performance.

        Calls best_setups() and worst_setups() (each requiring >= 5 evaluated
        signals) and converts them into human-readable "Prioritize / Avoid"
        strings.

        Returns:
            {
                "recommended": ["Prioritize A-class pullbacks (accumulation) ...", ...],
                "avoid":       ["Avoid B-class sideways (watchlist) ...", ...],
                "has_data":    bool   # False when not enough history yet
            }

        The caller is responsible for display — this method is analytics only.
        """
        best  = self.best_setups(min_evaluated=5, limit=5)
        worst = self.worst_setups(min_evaluated=5, limit=5)

        def _describe(v: dict, action: str) -> str:
            parts    = v["key"].split("|")
            cls, ms, ot, pri = (parts + ["?"] * 4)[:4]
            ar       = v["avg_returns"]
            ret_24h  = ar.get("24h")
            ret_str  = ("{:+.2f}%".format(ret_24h)) if ret_24h is not None else "n/a"
            wr_str   = "{:.0f}%".format(v["win_rate"])
            return (
                action + " " + cls + "-class " + ms.replace("_", " ")
                + " (" + ot.replace("_", " ") + ")"
                + "  WR:" + wr_str + "  24h:" + ret_str
                + "  [" + str(v["evaluated"]) + " eval]"
            )

        recommended = [_describe(v, "Prioritize") for v in best]
        avoid       = [_describe(v, "Avoid")      for v in worst]

        return {
            "recommended": recommended,
            "avoid":       avoid,
            "has_data":    bool(recommended or avoid),
        }


    def _opportunity_distribution(self) -> dict:
        """Count signals by opportunity_type across all logged signals."""
        counts: dict[str, int] = {label: 0 for label in _OPP_LABELS}
        for item in self._data["signals"]:
            ot = item.get("opportunity_type", "")
            if ot in counts:
                counts[ot] = counts[ot] + 1
            elif ot:
                counts[ot] = counts.get(ot, 0) + 1
        return counts

    def _opportunity_performance(self) -> dict:
        """Compute win rate and avg returns per opportunity_type.

        Same pattern as _class_performance().  Old signals without
        opportunity_type are excluded (cannot be backfilled — the type
        depends on real-time price state at signal generation time).
        """
        buckets: dict[str, list] = {label: [] for label in _OPP_LABELS}
        for item in self._data["signals"]:
            ot = item.get("opportunity_type", "")
            if not ot:
                continue
            buckets.setdefault(ot, []).append(item)

        result: dict[str, dict] = {}
        for ot, signals in buckets.items():
            if not signals:
                result[ot] = {
                    "total": 0, "evaluated": 0, "win_rate": 0.0,
                    "avg_returns": {h: None for h in EVALUATION_HORIZONS},
                }
                continue
            latest_changes = [self._latest_change(s) for s in signals]
            completed = [c for c in latest_changes if c is not None]
            winners   = sum(1 for c in completed if c > 0)
            avg_returns = {}
            for horizon in EVALUATION_HORIZONS:
                vals = [
                    self._to_float(s.get("evaluations", {}).get(horizon, {}).get("change_percent"))
                    for s in signals
                    if horizon in s.get("evaluations", {})
                ]
                avg_returns[horizon] = average(vals) if vals else None
            result[ot] = {
                "total":       len(signals),
                "evaluated":   len(completed),
                "win_rate":    (winners / len(completed) * 100) if completed else 0.0,
                "avg_returns": avg_returns,
            }
        return result

    def _market_state_distribution(self) -> dict:
        """Count signals by market_state across all logged signals."""
        states = ["bull_trend", "pullback", "recovery", "breakout", "sideways", "downtrend"]
        counts: dict[str, int] = {s: 0 for s in states}
        for item in self._data["signals"]:
            ms = item.get("market_state", "unknown")
            counts[ms] = counts.get(ms, 0) + 1
        return counts

    def _class_distribution(self) -> dict:
        """Count signals by coin_class across all logged signals.

        Old records without coin_class are classified live via get_coin_class()
        so the count is always complete, including historical signals.
        """
        counts: dict[str, int] = {"A": 0, "B": 0, "C": 0}
        for item in self._data["signals"]:
            cls = item.get("coin_class") or get_coin_class(item.get("coin", ""))
            counts[cls] = counts.get(cls, 0) + 1
        return counts


    def _class_performance(self) -> dict:
        """Compute signal count, win rate, and avg returns per coin class.

        Groups all logged signals by class (A / B / C), backfilling the class
        for old records via get_coin_class(coin).  Returns a dict keyed by
        class label, each containing:
          total       – signal count
          evaluated   – signals with a completed latest evaluation
          win_rate    – % of evaluated with positive latest return
          avg_returns – {horizon: float|None} for 1h, 4h, 24h
        """
        buckets: dict[str, list] = {"A": [], "B": [], "C": []}

        for item in self._data["signals"]:
            cls = item.get("coin_class") or get_coin_class(item.get("coin", ""))
            buckets.setdefault(cls, []).append(item)

        result: dict[str, dict] = {}
        for cls, signals in buckets.items():
            if not signals:
                result[cls] = {
                    "total": 0, "evaluated": 0, "win_rate": 0.0,
                    "avg_returns": {h: None for h in EVALUATION_HORIZONS},
                }
                continue

            latest_changes = [self._latest_change(s) for s in signals]
            completed = [c for c in latest_changes if c is not None]
            winners   = sum(1 for c in completed if c > 0)

            avg_returns = {}
            for horizon in EVALUATION_HORIZONS:
                vals = [
                    self._to_float(s.get("evaluations", {}).get(horizon, {}).get("change_percent"))
                    for s in signals
                    if horizon in s.get("evaluations", {})
                ]
                avg_returns[horizon] = average(vals) if vals else None

            result[cls] = {
                "total":       len(signals),
                "evaluated":   len(completed),
                "win_rate":    (winners / len(completed) * 100) if completed else 0.0,
                "avg_returns": avg_returns,
            }

        return result

    def _score_group_stats(self) -> dict:
        """Score distribution with new 60-69 / 70-79 / 80+ buckets.

        The legacy 50-59 bucket is retained for historical signals but will
        not receive new entries under the raised threshold.
        """
        groups = {"50-59": [], "60-69": [], "70-79": [], "80+": []}

        for item in self._data["signals"]:
            score = int(self._to_float(item.get("score")))
            change = self._latest_change(item)
            if change is None:
                continue

            if 50 <= score <= 59:
                groups["50-59"].append(change)
            elif 60 <= score <= 69:
                groups["60-69"].append(change)
            elif 70 <= score <= 79:
                groups["70-79"].append(change)
            elif score >= 80:
                groups["80+"].append(change)

        return {
            label: {
                "count": len(values),
                "win_rate": (sum(1 for value in values if value > 0) / len(values) * 100) if values else 0.0,
            }
            for label, values in groups.items()
        }


    def coin_performance(self, top_n: int = 10) -> list[dict]:
        """Aggregate per-coin stats from signals.json history.

        For each coin that has at least one logged signal, computes:
          - total_signals    : how many signals were logged
          - evaluated        : how many have a completed 24h evaluation
          - win_rate         : % of evaluated signals where 24h return > 0
          - avg_return_24h   : mean 24h return across evaluated signals
          - avg_final_score  : mean final_score across all signals for the coin

        Returns top_n coins ranked by a composite key:
          (win_rate * 0.6 + normalised avg_return_24h * 0.4)
        so that both consistency (win rate) and magnitude (return) matter.
        """
        from collections import defaultdict as _dd

        buckets: dict = _dd(lambda: {
            "total": 0, "returns_24h": [], "final_scores": []
        })

        for item in self._data["signals"]:
            coin = item.get("coin")
            if not coin:
                continue
            b = buckets[coin]
            b["total"] += 1
            b["final_scores"].append(self._to_float(item.get("final_score", 0)))

            eval_24h = item.get("evaluations", {}).get("24h")
            if eval_24h is not None:
                b["returns_24h"].append(
                    self._to_float(eval_24h.get("change_percent"))
                )

        rows: list[dict] = []
        for coin, b in buckets.items():
            evaluated   = len(b["returns_24h"])
            wins        = sum(1 for r in b["returns_24h"] if r > 0)
            win_rate    = (wins / evaluated * 100) if evaluated else 0.0
            avg_ret     = average(b["returns_24h"]) if b["returns_24h"] else None
            avg_score   = average(b["final_scores"]) if b["final_scores"] else 0.0

            rows.append({
                "coin":            coin,
                "total_signals":   b["total"],
                "evaluated":       evaluated,
                "win_rate":        win_rate,
                "avg_return_24h":  avg_ret,
                "avg_final_score": round(avg_score, 1),
            })

        # Composite rank: win_rate (60%) + avg_return_24h normalised (40%)
        # Use 0 for coins with no completed evaluations so they sink to bottom
        def _rank_key(row: dict) -> float:
            wr  = row["win_rate"]                              # already 0-100
            ret = row["avg_return_24h"] if row["avg_return_24h"] is not None else 0.0
            ret_norm = max(-100.0, min(100.0, ret))           # clamp to ±100
            return wr * 0.6 + ret_norm * 0.4

        rows.sort(key=_rank_key, reverse=True)
        return rows[:top_n]

    def daily_summary(self) -> dict:
        """Compute summary stats for signals logged in the last 24 hours.

        Returns a dict with:
            signals  – count of signals in the last 24 h
            wins     – signals with positive latest return
            losses   – signals with negative/zero latest return
            win_rate – wins / evaluated * 100  (0.0 when no evaluations)
            avg_1h   – average 1h return (None when no data)
            avg_4h   – average 4h return (None when no data)
            avg_24h  – average 24h return (None when no data)
            top_coin – coin with highest final_score today (None when no signals)

        Never raises — returns safe defaults on any error.
        """
        try:
            from datetime import timezone as _tz, timedelta as _td
            cutoff = datetime.now(_tz.utc) - _td(hours=24)
            cutoff_str = cutoff.isoformat()

            today_signals = [
                s for s in self._data["signals"]
                if s.get("timestamp", "") >= cutoff_str
            ]

            if not today_signals:
                return {
                    "signals": 0, "wins": 0, "losses": 0, "win_rate": 0.0,
                    "avg_1h": None, "avg_4h": None, "avg_24h": None,
                    "top_coin": None,
                }

            # Evaluation-based stats
            wins = losses = 0
            returns: dict[str, list] = {"1h": [], "4h": [], "24h": []}

            for item in today_signals:
                change = self._latest_change(item)
                if change is not None:
                    if change > 0:
                        wins += 1
                    else:
                        losses += 1
                for h in ("1h", "4h", "24h"):
                    val = item.get("evaluations", {}).get(h, {}).get("change_percent")
                    if val is not None:
                        returns[h].append(self._to_float(val))

            evaluated = wins + losses
            win_rate  = (wins / evaluated * 100) if evaluated else 0.0

            def _avg(vals):
                return round(sum(vals) / len(vals), 4) if vals else None

            # Top coin by final_score today
            top = max(
                today_signals,
                key=lambda s: self._to_float(s.get("final_score", 0)),
                default=None,
            )
            top_coin = top.get("coin") if top else None

            return {
                "signals":  len(today_signals),
                "wins":     wins,
                "losses":   losses,
                "win_rate": round(win_rate, 2),
                "avg_1h":   _avg(returns["1h"]),
                "avg_4h":   _avg(returns["4h"]),
                "avg_24h":  _avg(returns["24h"]),
                "top_coin": top_coin,
            }
        except Exception:
            return {
                "signals": 0, "wins": 0, "losses": 0, "win_rate": 0.0,
                "avg_1h": None, "avg_4h": None, "avg_24h": None,
                "top_coin": None,
            }


    def _latest_change(self, item: dict) -> Optional[float]:
        evaluations = item.get("evaluations", {})
        for label in ("24h", "4h", "1h"):
            if label in evaluations:
                return self._to_float(evaluations[label].get("change_percent"))
        return None

    def _is_duplicate(self, signal: Signal) -> bool:
        signal_minute = signal.created_at.replace(second=0, microsecond=0).isoformat()
        for item in self._data["signals"][-50:]:
            item_time = self._parse_time(item.get("timestamp"))
            if item_time is None:
                continue
            item_minute = item_time.replace(second=0, microsecond=0).isoformat()
            if item.get("coin") == signal.coin and item.get("score") == signal.score and item_minute == signal_minute:
                return True
        return False

    @staticmethod
    def _parse_time(value) -> Optional[datetime]:
        if not value:
            return None
        try:
            parsed = datetime.fromisoformat(str(value))
        except ValueError:
            return None
        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=timezone.utc)
        return parsed

    @staticmethod
    def _to_float(value) -> float:
        try:
            return float(value)
        except (TypeError, ValueError):
            return 0.0

# =============================================================================
# ASYNC SCANNER
# =============================================================================
# The scanner fetches all tickers once, then analyzes coins concurrently.

class Scanner:
    def __init__(
        self,
        watchlist_store: WatchlistStore,
        alert_callback: Callable[[Signal, str], object],
        performance_tracker: SignalPerformanceTracker,
        client: Optional[CoinDCXPublicClient] = None,
    ):
        self.watchlist_store = watchlist_store
        self.alert_callback = alert_callback
        self.performance_tracker = performance_tracker
        self.client = client or CoinDCXPublicClient()

        self.price_history: dict[str, list[dict]] = defaultdict(list)
        self.last_alert_at: dict[str, datetime] = {}

        self._alert_in_flight: set[str] = set()
        self._alert_lock = asyncio.Lock()
        self._scan_semaphore = asyncio.Semaphore(SCAN_CONCURRENCY)

        self._ticker_cache: Optional[list[dict]] = None
        self._ticker_cache_at = 0.0
        self._ticker_lock = asyncio.Lock()

        # Populated by run_bootstrap(); read by /bootstrap command
        self._bootstrap_result: Optional[BootstrapResult] = None

    async def run_bootstrap(self) -> BootstrapResult:
        """Pre-fill price_history for all known coins from CoinDCX 5-min candles.

        Called from post_init before run_forever() begins so the first scan
        cycle has full EMA / MTF / Phase 5 history available immediately.

        Coin list = watchlist ∪ (top discovery coins from current tickers).
        Using both sets ensures both watched and discovered coins are warm.
        """
        if not BOOTSTRAP_ENABLED:
            logger.info("Bootstrap disabled (BOOTSTRAP_ENABLED=false)")
            self._bootstrap_result = BootstrapResult()
            return self._bootstrap_result

        logger.info("Bootstrap: fetching current tickers to build coin list...")
        try:
            tickers = await self.get_tickers(force=True)
        except Exception:
            logger.warning("Bootstrap: ticker fetch failed; skipping bootstrap", exc_info=True)
            self._bootstrap_result = BootstrapResult()
            return self._bootstrap_result

        # Build coin list: watchlist + discovery candidates (up to limit)
        ticker_map    = self._ticker_map(tickers)
        watchlist_set = set(self.watchlist_store.all())
        discovery_set = {
            coin for coin, ticker in ticker_map.items()
            if coin not in watchlist_set and self._passes_discovery_filters(ticker)
        }
        all_coins = list(watchlist_set) + list(discovery_set)[:DISCOVERY_MAX_COINS]

        logger.info(
            "Bootstrap: loading history for %d coins (%d watchlist + %d discovery)",
            len(all_coins), len(watchlist_set), len(all_coins) - len(watchlist_set),
        )

        result = await bootstrap_price_history(
            coins=all_coins,
            price_history=self.price_history,
            concurrency=BOOTSTRAP_CONCURRENCY,
        )
        self._bootstrap_result = result
        return result

    async def run_forever(self) -> None:
        discovery_due = 0.0
        logger.info(
            "Scanner started: scan_interval=%ss discovery_interval=%ss concurrency=%s",
            SCAN_INTERVAL_SECONDS,
            DISCOVERY_INTERVAL_SECONDS,
            SCAN_CONCURRENCY,
        )

        while True:
            started = asyncio.get_running_loop().time()
            try:
                tickers = await self.get_tickers(force=True)
                self.evaluate_signal_performance(tickers)
                watchlist_signals = await self.scan_watchlist(tickers)

                now = asyncio.get_running_loop().time()
                if now >= discovery_due:
                    discovery_signals = await self.scan_market(tickers)
                    discovery_due = now + DISCOVERY_INTERVAL_SECONDS
                else:
                    discovery_signals = []

                elapsed = asyncio.get_running_loop().time() - started
                logger.info(
                    "Scan complete: watchlist_signals=%s discovery_signals=%s elapsed=%.2fs",
                    len(watchlist_signals),
                    len(discovery_signals),
                    elapsed,
                )
            except Exception:
                logger.exception("Scanner loop failed; retrying next interval")

            elapsed = asyncio.get_running_loop().time() - started
            await asyncio.sleep(max(5, SCAN_INTERVAL_SECONDS - elapsed))

    async def get_tickers(self, force: bool = False) -> list[dict]:
        async with self._ticker_lock:
            now = asyncio.get_running_loop().time()
            cache_fresh = (
                self._ticker_cache is not None
                and now - self._ticker_cache_at < TICKER_CACHE_TTL_SECONDS
            )
            if cache_fresh and not force:
                return self._ticker_cache or []

            try:
                tickers = await asyncio.to_thread(self.client.fetch_tickers)
            except (requests.RequestException, ValueError):
                if self._ticker_cache is not None:
                    logger.warning("Ticker fetch failed; using cached data", exc_info=True)
                    return self._ticker_cache
                raise

            self._ticker_cache = tickers
            self._ticker_cache_at = asyncio.get_running_loop().time()
            logger.debug("Fetched %s ticker rows", len(tickers))
            return tickers

    def evaluate_signal_performance(self, tickers: list[dict]) -> None:
        ticker_map = self._ticker_map(tickers)
        prices = {
            coin: self._extract_price_volume(ticker)[0]
            for coin, ticker in ticker_map.items()
        }
        updated = self.performance_tracker.evaluate_due_signals(prices)
        if updated:
            logger.info("Updated %s signal performance checkpoints", updated)

    async def scan_watchlist(self, tickers: Optional[list[dict]] = None) -> list[Signal]:
        tickers = tickers or await self.get_tickers()
        ticker_map = self._ticker_map(tickers)
        coins = [coin for coin in self.watchlist_store.all() if coin in ticker_map]
        return await self._scan_many(coins, ticker_map, source="watchlist")

    async def scan_market(self, tickers: Optional[list[dict]] = None) -> list[Signal]:
        tickers = tickers or await self.get_tickers()
        ticker_map = self._ticker_map(tickers)
        watchlist = set(self.watchlist_store.all())
        coins = [
            coin
            for coin, ticker in ticker_map.items()
            if coin not in watchlist and self._passes_discovery_filters(ticker)
        ][:DISCOVERY_MAX_COINS]
        return await self._scan_many(coins, ticker_map, source="discovery")

    async def coin_snapshot(self, coin: str) -> Optional[dict]:
        coin = coin.upper().strip()
        tickers = await self.get_tickers()
        ticker = self._ticker_map(tickers).get(coin)
        if not ticker:
            return None

        price, volume = self._extract_price_volume(ticker)
        if price <= 0:
            return None

        self._append_history(coin, price, volume)
        history = self.price_history[coin]
        return {
            "coin": coin,
            "price": price,
            "volume": volume,
            "history_count": len(history),
            "trend": trend_summary(history),
            "signals": analyze_coin(coin, history),
        }

    async def _scan_many(
        self,
        coins: list[str],
        ticker_map: dict[str, dict],
        source: str,
    ) -> list[Signal]:
        tasks = [
            asyncio.create_task(self._scan_ticker_bounded(coin, ticker_map[coin], source))
            for coin in coins
        ]
        if not tasks:
            return []

        results = await asyncio.gather(*tasks, return_exceptions=True)
        signals: list[Signal] = []
        for coin, result in zip(coins, results):
            if isinstance(result, Exception):
                logger.error(
                    "Coin scan failed: coin=%s source=%s",
                    coin,
                    source,
                    exc_info=(type(result), result, result.__traceback__),
                )
                continue
            signals.extend(result)

        top_signals = self._rank_signals(signals)[:MAX_RESULTS]
        for signal in top_signals:
            if not smart_filter(signal):
                continue
            if not learning_filter(signal, self.performance_tracker):
                continue
            if not historical_filter(signal):
                continue
            await self._send_alert_once(signal, source)
        return top_signals

    async def _scan_ticker_bounded(self, coin: str, ticker: dict, source: str) -> list[Signal]:
        async with self._scan_semaphore:
            return await self._scan_ticker(coin, ticker, source)

    async def _scan_ticker(self, coin: str, ticker: dict, source: str) -> list[Signal]:
        price, volume = self._extract_price_volume(ticker)
        if price <= 0:
            logger.debug("Skipping %s because price is invalid", coin)
            return []

        self._append_history(coin, price, volume)
        return analyze_coin(coin, self.price_history[coin])

    async def _send_alert_once(self, signal: Signal, source: str) -> None:
        async with self._alert_lock:
            if signal.coin in self._alert_in_flight:
                return
            if not self._cooldown_passed(signal.coin):
                return
            self._alert_in_flight.add(signal.coin)

        sent = False
        try:
            self.performance_tracker.log_signal(signal)
            await self.alert_callback(signal, source)
            sent = True
        except Exception:
            logger.exception("Failed to send alert for %s", signal.coin)
        finally:
            async with self._alert_lock:
                if sent:
                    self.last_alert_at[signal.coin] = datetime.now(timezone.utc)
                self._alert_in_flight.discard(signal.coin)

        if sent:
            logger.info(
                "Alert sent: coin=%s kind=%s source=%s score=%s",
                signal.coin,
                signal.kind,
                source,
                signal.score,
            )

    def _append_history(self, coin: str, price: float, volume: float) -> None:
        history = self.price_history[coin]
        history.append(
            {
                "time": datetime.now(timezone.utc),
                "price": price,
                "volume": volume,
            }
        )
        del history[:-PRICE_HISTORY_LIMIT]

    def _cooldown_passed(self, coin: str) -> bool:
        previous = self.last_alert_at.get(coin)
        if previous is None:
            return True
        age = (datetime.now(timezone.utc) - previous).total_seconds()
        return age >= ALERT_COOLDOWN_SECONDS

    def _ticker_map(self, tickers: list[dict]) -> dict[str, dict]:
        pairs: dict[str, dict] = {}
        priorities = {quote: index for index, quote in enumerate(QUOTE_PRIORITY)}
        selected_priority: dict[str, int] = {}

        for ticker in tickers:
            market = str(ticker.get("market", "")).upper()
            for quote in QUOTE_PRIORITY:
                if market.endswith(quote) and len(market) > len(quote):
                    coin = market[: -len(quote)]
                    priority = priorities[quote]
                    if coin not in pairs or priority < selected_priority[coin]:
                        pairs[coin] = ticker
                        selected_priority[coin] = priority
                    break

        return pairs

    def _passes_discovery_filters(self, ticker: dict) -> bool:
        """Stricter discovery filters to eliminate low-cap / weak coins.

        Requires:
        * price  >= MIN_PRICE
        * 24h volume >= MIN_VOLUME_24H  (raised from 100k to 500k)
        * 24h quote volume (liquidity proxy) >= MIN_LIQUIDITY_24H
        """
        price, volume = self._extract_price_volume(ticker)
        if price < MIN_PRICE:
            return False
        if volume < MIN_VOLUME_24H:
            return False
        # Liquidity gate: use quote_volume when available, else volume * price
        quote_vol = self._to_float(
            ticker.get("quote_volume") or ticker.get("volume_24h")
        )
        if quote_vol <= 0:
            quote_vol = volume * price
        if quote_vol < MIN_LIQUIDITY_24H:
            return False
        return True

    @staticmethod
    def _rank_signals(signals: list[Signal]) -> list[Signal]:
        """Rank by final_score (blend of scanner + Phase 5), then phase5_total."""
        return sorted(
            signals,
            key=lambda signal: (signal.final_score, signal.phase5_total, signal.hist_total, signal.score),
            reverse=True,
        )

    def _extract_price_volume(self, ticker: dict) -> tuple[float, float]:
        price = self._to_float(ticker.get("last_price"))
        volume = self._to_float(
            ticker.get("volume")
            or ticker.get("volume_24h")
            or ticker.get("quote_volume")
            or ticker.get("base_volume")
        )
        return price, volume

    @staticmethod
    def _to_float(value) -> float:
        try:
            return float(value)
        except (TypeError, ValueError):
            return 0.0


# =============================================================================
# TELEGRAM BOT COMMANDS
# =============================================================================

class TelegramAlertBot:
    def __init__(self, watchlist_store: WatchlistStore, performance_tracker: SignalPerformanceTracker):
        if not BOT_TOKEN:
            raise RuntimeError("Set TELEGRAM_BOT_TOKEN before starting the bot.")

        self.watchlist_store = watchlist_store
        self.performance_tracker = performance_tracker
        self.scanner: Optional[Scanner] = None
        self.chat_ids: set[int] = set()
        self.app: Application = (
            ApplicationBuilder()
            .token(BOT_TOKEN)
            .connect_timeout(TG_CONNECT_TIMEOUT)
            .read_timeout(TG_READ_TIMEOUT)
            .write_timeout(TG_WRITE_TIMEOUT)
            .pool_timeout(TG_POOL_TIMEOUT)
            .build()
        )
        self._register_commands()

    def set_scanner(self, scanner: Scanner) -> None:
        self.scanner = scanner

    async def send_alert(self, signal: Signal, source: str) -> None:
        if not self.chat_ids:
            return

        emoji = "💎" if signal.tier == "PREMIUM" else ("🚀" if signal.final_score >= 70 else "📈")
        score_label = "80+" if signal.score >= 80 else ("70-79" if signal.score >= 70 else "60-69")
        text = (
            f"{emoji} {signal.tier} [{score_label}]\n\n"
            f"Coin: {signal.coin}\n"
            f"Class: {signal.coin_class}  |  Market: {signal.market_state}\n"
            f"Opportunity: {signal.opportunity_type}\n\n"
            f"Confidence: {signal.opp_confidence}\n"
            f"Score: {signal.opportunity_score}\n"
            f"Risk: {signal.risk_level.capitalize()}\n"
            f"Priority: {signal.priority}\n\n"
            f"Price: {format_price(signal.price)}\n"
            f"24h Volume: {format_volume(signal.volume)}\n\n"
            f"── Scores ──\n"
            f"Scanner    : {signal.score}\n"
            f"Phase5     : {signal.phase5_total}  "
            f"(TQ={signal.phase5_trend} PQ={signal.phase5_pullback} "
            f"MOM={signal.phase5_momentum} RR={signal.phase5_risk_reward})\n"
            f"Historical : {signal.hist_total}  "
            f"(7d={signal.hist_trend_7d} 30d={signal.hist_trend_30d} "
            f"90d={signal.hist_trend_90d} SR={signal.hist_sr_quality} "
            f"Vol={signal.hist_vol_score})\n"
            f"Final      : {signal.final_score}\n\n"
            f"── Reasons ──\n"
            f"{self._format_reasons(signal)}\n\n"
            f"Vol strength: {signal.volume_strength:.2f}x  |  Model: {signal.model_version}"
        )
        for chat_id in list(self.chat_ids):
            await self.app.bot.send_message(chat_id=chat_id, text=text)

    async def start(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        self.chat_ids.add(update.effective_chat.id)
        await update.message.reply_text(
            "Crypto alert scanner is active.\n"
            "Commands: /watchlist, /add COIN, /remove COIN, /scan, /market, /top, /price COIN, /trend COIN, /stats, /history, /coins, /daily, /mtf, /bootstrap, /health, /storage"
        )

    async def show_watchlist(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        coins = self.watchlist_store.all()
        await update.message.reply_text("Watchlist:\n" + "\n".join(f"- {coin}" for coin in coins))

    async def add_coin(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        if not context.args:
            await update.message.reply_text("Usage: /add SOL")
            return

        coin = context.args[0]
        added = self.watchlist_store.add(coin)
        await update.message.reply_text(
            f"Added {coin.upper()}" if added else f"{coin.upper()} is already on the watchlist"
        )

    async def remove_coin(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        if not context.args:
            await update.message.reply_text("Usage: /remove SOL")
            return

        coin = context.args[0]
        removed = self.watchlist_store.remove(coin)
        await update.message.reply_text(
            f"Removed {coin.upper()}" if removed else f"{coin.upper()} is not on the watchlist"
        )

    async def scan(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        if self.scanner is None:
            await update.message.reply_text("Scanner is starting. Try again in a moment.")
            return


        signals = await self.scanner.scan_watchlist()
        print("Signals Returned =", signals)
        print("Count =", len(signals))

        # NEW: Share signals with MTB API
        global LATEST_MTB_SIGNALS
        LATEST_MTB_SIGNALS = []

        for s in signals:
            LATEST_MTB_SIGNALS.append({
        "strategy": "MTB",
        "symbol": s.coin,
        "action": "BUY",
        "entry": s.price,
        "stop_loss": round(s.price * 0.97, 4),
        "take_profit": round(s.price * 1.06, 4),
        "signal_time": datetime.utcnow().isoformat() + "Z"
    })

        if not signals:
           await update.message.reply_text("Watchlist scanned. No active signals.")
           return

        await update.message.reply_text(self._format_signal_summary(signals))



    async def market(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        if self.scanner is None:
            await update.message.reply_text("Scanner is starting. Try again in a moment.")
            return

        signals = await self.scanner.scan_market()
        if not signals:
            await update.message.reply_text("Market scanned. No active discovery signals.")
            return
        await update.message.reply_text(self._format_signal_summary(signals))

    async def top(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        """Show top 10 stored signals ranked by opportunity_score."""
        await self._refresh_due_performance()
        ranked = self.performance_tracker.top_ranked_signals(limit=10)
        if not ranked:
            await update.message.reply_text(
                "No ranked signals yet.\n"
                "Signals from v12+ will appear here once generated."
            )
            return

        lines = ["\U0001f3c6 Top Opportunities\n" + "\u2501" * 19]
        for idx, item in enumerate(ranked, 1):
            coin      = item.get("coin", "?")
            cls_tag   = item.get("coin_class", "?")
            ms_tag    = item.get("market_state", "?")
            opp_tag   = item.get("opportunity_type", "?")
            opp_sc    = item.get("opportunity_score", "?")
            conf_val  = item.get("opp_confidence", "?")
            risk_val  = item.get("risk_level", "?").capitalize() if item.get("risk_level") else "?"
            pri_val   = item.get("priority", "?")
            ts        = item.get("timestamp", "")[:10]   # date only
            entry = (
                "\n" + str(idx) + ". " + coin + " (" + cls_tag + ")  " + ts + "\n"
                "   " + ms_tag + " → " + opp_tag + "\n"
                "   Score: " + str(opp_sc)
                + "  Conf: " + str(conf_val)
                + "  Risk: " + risk_val + "\n"
                "   Priority: " + pri_val
            )
            lines.append(entry)

        await update.message.reply_text("\n".join(lines))

    async def price(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        snapshot = await self._snapshot_from_args(update, context, usage="Usage: /price SOL")
        if snapshot is None:
            return
        await update.message.reply_text(
            f"{snapshot['coin']} price: {format_price(snapshot['price'])}\n"
            f"24h Volume: {format_volume(snapshot['volume'])}"
        )

    async def stats(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        await self._refresh_due_performance()
        self.performance_tracker.save_stats()
        stats = self.performance_tracker.stats()
        averages = stats["average_returns"]
        groups = stats["score_groups"]
        mv = stats["model_stats"].get(MODEL_VERSION, {})
        mv_avg = mv.get("average_returns", {})

        mtf_d = stats.get("mtf_diagnostics", {})
        total_mtf = sum(mtf_d.values()) or 1   # avoid div-by-zero

        def _pct(n): return f"{n / total_mtf * 100:.0f}%"

        # ── Overall (all versions) ──────────────────────────────────────────
        overall = (
            "📊 Signal Performance Stats\n"
            "━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"Total signals (all): {stats['total_signals']}\n"
            f"Winning: {stats['winning_signals']}  Losing: {stats['losing_signals']}\n"
            f"Win rate: {stats['win_rate']:.2f}%\n\n"
            "Avg returns (all):\n"
            f"  1h: {format_percent(averages.get('1h'))}\n"
            f"  4h: {format_percent(averages.get('4h'))}\n"
            f"  24h: {format_percent(averages.get('24h'))}\n\n"
            "Score distribution:\n"
            f"  50-59 (legacy): {groups['50-59']['win_rate']:.1f}% ({groups['50-59']['count']})\n"
            f"  60-69: {groups['60-69']['win_rate']:.1f}% ({groups['60-69']['count']})\n"
            f"  70-79: {groups['70-79']['win_rate']:.1f}% ({groups['70-79']['count']})\n"
            f"  80+:   {groups['80+']['win_rate']:.1f}% ({groups['80+']['count']})\n\n"
            "Filtered this session:\n"
            f"  No MTF confirm: {stats['filtered_no_mtf']}\n"
            f"  No EMA crossover: {stats['filtered_no_ema']}\n"
            f"  No volume spike: {stats['filtered_no_volume']}\n"
            f"  Low score (<60): {stats['filtered_low_score']}"
        )

        # ── Coin class distribution ─────────────────────────────────────────
        # Opportunity type distribution + performance
        opp_dist = stats.get("opportunity_distribution", {})
        opp_perf = stats.get("opportunity_performance", {})
        opp_order = ["accumulation", "recovery_trade", "momentum_trade",
                     "continuation", "watchlist", "avoid"]

        def _opp_row(ot: str) -> str:
            n  = opp_dist.get(ot, 0)
            p  = opp_perf.get(ot, {})
            ev = p.get("evaluated", 0)
            wr = p.get("win_rate", 0.0)
            ar = p.get("avg_returns", {})
            label = ot.replace("_", " ").title()
            if ev:
                return ("  " + label + ": " + str(n)
                        + "  |  " + str(ev) + " eval"
                        + "  WR:" + "{:.0f}%".format(wr)
                        + "  1h:" + format_percent(ar.get("1h"))
                        + "  24h:" + format_percent(ar.get("24h")))
            return "  " + label + ": " + str(n) + ("  (no evals yet)" if n else "")

        opp_section = (
            "\n\nOpportunity Types\n"
            + "=" * 27 + "\n"
            + "\n".join(_opp_row(ot) for ot in opp_order)
        )

        # Priority distribution + performance
        pri_dist = stats.get("priority_distribution", {})
        pri_perf = stats.get("priority_performance", {})
        pri_order = [label for _, label in PRIORITY_LEVELS]  # Elite→High→Medium→Watch→Ignore

        def _pri_row(label: str) -> str:
            n  = pri_dist.get(label, 0)
            p  = pri_perf.get(label, {})
            ev = p.get("evaluated", 0)
            wr = p.get("win_rate", 0.0)
            ar = p.get("avg_returns", {})
            if ev:
                return (
                    "  " + label + ": " + str(n)
                    + "  |  " + str(ev) + " eval"
                    + "  WR:" + "{:.0f}%".format(wr)
                    + "  1h:" + format_percent(ar.get("1h"))
                    + "  4h:" + format_percent(ar.get("4h"))
                    + "  24h:" + format_percent(ar.get("24h"))
                )
            return "  " + label + ": " + str(n) + ("  (no evals yet)" if n else "")

        priority_section = (
            "\n\n\U0001f4ca Priority Distribution\n"
            + "=" * 27 + "\n"
            + "\n".join(_pri_row(l) for l in pri_order)
        )

        # Learning recommendations
        learn = stats.get("learning_recommendations", {})
        if learn.get("has_data"):
            rec_lines  = ["  \u2022 " + s for s in learn.get("recommended", [])]
            avoid_lines = ["  \u2022 " + s for s in learn.get("avoid", [])]
            learn_body = (
                "Recommended:\n" + ("\n".join(rec_lines) if rec_lines else "  (none)") + "\n\n"
                + "Avoid:\n"      + ("\n".join(avoid_lines) if avoid_lines else "  (none)")
            )
        else:
            learn_body = "  Not enough evaluated signals yet"
        learn_section = (
            "\n\n\U0001f9e0 Learning Recommendations\n"
            + "=" * 27 + "\n"
            + learn_body
        )

        # Market state distribution
        ms_dist = stats.get("market_state_distribution", {})
        ms_order = ["bull_trend", "breakout", "recovery", "pullback", "sideways", "downtrend"]
        ms_lines = []
        for ms in ms_order:
            n_ms = ms_dist.get(ms, 0)
            if n_ms:
                ms_lines.append("  " + ms.replace("_", " ").title() + ": " + str(n_ms))
        market_state_section = (
            "\n\nMarket States\n"
            + "=" * 27 + "\n"
            + ("\n".join(ms_lines) if ms_lines else "  No signals yet")
        )

        # Smart filter analytics
        f_smart  = stats.get("filtered_smart", 0)
        f_learn  = stats.get("filtered_learning", 0)
        f_hist   = stats.get("filtered_historical", 0)
        f_total  = f_smart + f_learn + f_hist
        smart_section = (
            "\n\n\U0001f6e1\ufe0f Smart Filtering\n"
            + "=" * 27 + "\n"
            + "  Smart Filter    : " + str(f_smart) + "\n"
            + "  Learning Filter : " + str(f_learn) + "\n"
            + "  Historical Filter: " + str(f_hist) + "\n\n"
            + "  Total Filtered  : " + str(f_total)
        )

        cls_dist = stats.get("class_distribution", {})
        cls_perf = stats.get("class_performance", {})

        def _cls_row(label: str) -> str:
            p = cls_perf.get(label, {})
            total = p.get("total", 0)
            evaluated = p.get("evaluated", 0)
            wr = p.get("win_rate", 0.0)
            ar = p.get("avg_returns", {})
            r1h  = format_percent(ar.get("1h"))
            r4h  = format_percent(ar.get("4h"))
            r24h = format_percent(ar.get("24h"))
            return (
                "  Class " + label + ": "
                + str(total) + " signals"
                + ("  |  " + str(evaluated) + " eval  WR:" + "{:.0f}%".format(wr)
                   + "  1h:" + r1h + "  4h:" + r4h + "  24h:" + r24h
                   if evaluated else "  (no evaluations yet)")
            )

        class_section = (
            "\n\n📊 Coin Class Performance\n"
            + "━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            + _cls_row("A") + "\n"
            + _cls_row("B") + "\n"
            + _cls_row("C")
        )

        # ── MTF diagnostics ─────────────────────────────────────────────────
        mtf_section = (
            "\n\n📡 MTF Diagnostics (this session)\n"
            "━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"  5m only    : {mtf_d.get('5m_only', 0):4d}  ({_pct(mtf_d.get('5m_only', 0))})\n"
            f"  15m only   : {mtf_d.get('15m_only', 0):4d}  ({_pct(mtf_d.get('15m_only', 0))})\n"
            f"  5m + 15m   : {mtf_d.get('5m_15m', 0):4d}  ({_pct(mtf_d.get('5m_15m', 0))})\n"
            f"  5m+15m+1h  : {mtf_d.get('5m_15m_1h', 0):4d}  ({_pct(mtf_d.get('5m_15m_1h', 0))})\n"
            f"  No align   : {mtf_d.get('none', 0):4d}  ({_pct(mtf_d.get('none', 0))})"
        )

        # ── Current model ───────────────────────────────────────────────────
        current = (
            f"\n\n🤖 Current model: {MODEL_VERSION}\n"
            "━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"Signals: {mv.get('total', 0)}  Evaluated: {mv.get('evaluated', 0)}\n"
            f"Win rate: {mv.get('win_rate', 0.0):.2f}%\n\n"
            "Avg returns:\n"
            f"  1h: {format_percent(mv_avg.get('1h'))}\n"
            f"  4h: {format_percent(mv_avg.get('4h'))}\n"
            f"  24h: {format_percent(mv_avg.get('24h'))}"
        )

        await update.message.reply_text(
            overall + opp_section + priority_section
            + learn_section + smart_section
            + market_state_section + class_section + mtf_section + current
        )

    async def history(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        if context.args:
            snapshot = await self._snapshot_from_args(update, context, usage="Usage: /history SOL")
            if snapshot is None:
                return
            await update.message.reply_text(
                f"{snapshot['coin']} history samples: {snapshot['history_count']}"
            )
            return

        current_prices = await self._refresh_due_performance()
        signals = self.performance_tracker.recent_signals(limit=10, current_prices=current_prices)
        if not signals:
            await update.message.reply_text("No signals logged yet.")
            return

        lines = ["Last 10 signals:"]
        for item in signals:
            current = item.get("current_change_percent")
            latest = item.get("latest_change_percent")
            performance = current if current is not None else latest
            reasons = ", ".join(item.get("reasons", []))
            mv_tag     = item.get("model_version", "?")
            cls_tag    = item.get("coin_class", "?")
            ms_tag     = item.get("market_state", "?")
            opp_tag    = item.get("opportunity_type", "?")
            conf_val   = item.get("opp_confidence", "?")
            opp_sc     = item.get("opportunity_score", "?")
            risk_val   = item.get("risk_level", "?").capitalize() if item.get("risk_level") else "?"
            pri_val    = item.get("priority", "?")
            p5_total   = item.get("phase5", {}).get("total", "?")
            hist_total = item.get("historical_score", {}).get("total", "?")
            final_sc   = item.get("final_score", "?")
            lines.append(
                f"\n{item.get('coin')} | {cls_tag} | {ms_tag} | {opp_tag} | {pri_val} | {mv_tag}\n"
                f"Confidence: {conf_val}  Score: {opp_sc}  Risk: {risk_val}\n"
                f"Scanner:{item.get('score')} P5:{p5_total} H:{hist_total} F:{final_sc}\n"
                f"Signal price: {format_price(item.get('signal_price'))}\n"
                f"Current performance: {format_percent(performance)}\n"
                f"Reasons: {reasons}"
            )
        await update.message.reply_text("\n".join(lines))
    async def trend(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        snapshot = await self._snapshot_from_args(update, context, usage="Usage: /trend SOL")
        if snapshot is None:
            return

        trend = snapshot["trend"]
        await update.message.reply_text(
            f"{snapshot['coin']} trend: {trend['trend']}\n"
            f"Recent move: {trend['move_percent']:+.2f}%\n"
            f"EMA fast: {format_price(trend.get('ema_fast', 0))}\n"
            f"EMA slow: {format_price(trend.get('ema_slow', 0))}"
        )

    async def _snapshot_from_args(
        self,
        update: Update,
        context: ContextTypes.DEFAULT_TYPE,
        usage: str,
    ) -> Optional[dict]:
        if self.scanner is None:
            await update.message.reply_text("Scanner is starting. Try again in a moment.")
            return None
        if not context.args:
            await update.message.reply_text(usage)
            return None

        snapshot = await self.scanner.coin_snapshot(context.args[0])
        if snapshot is None:
            await update.message.reply_text(f"No public ticker found for {context.args[0].upper()}")
            return None
        return snapshot

    async def _current_price_map(self) -> dict[str, float]:
        if self.scanner is None:
            return {}
        tickers = await self.scanner.get_tickers()
        ticker_map = self.scanner._ticker_map(tickers)
        return {
            coin: self.scanner._extract_price_volume(ticker)[0]
            for coin, ticker in ticker_map.items()
        }

    async def _refresh_due_performance(self) -> dict[str, float]:
        current_prices = await self._current_price_map()
        self.performance_tracker.evaluate_due_signals(current_prices)
        return current_prices
    async def daily(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        """Show a summary of today's signal activity."""
        await self._refresh_due_performance()
        d = self.performance_tracker.daily_summary()

        def _fmt(val):
            return ("{:+.2f}%".format(val)) if val is not None else "n/a"

        lines = [
            "\U0001f4c5 Daily Summary",
            "=" * 27,
            "",
            "Signals : " + str(d["signals"]),
            "Wins    : " + str(d["wins"]),
            "Losses  : " + str(d["losses"]),
            "Win Rate: " + "{:.1f}%".format(d["win_rate"]),
            "",
            "Avg 1h  : " + _fmt(d["avg_1h"]),
            "Avg 4h  : " + _fmt(d["avg_4h"]),
            "Avg 24h : " + _fmt(d["avg_24h"]),
            "",
            "Top Coin: " + (d["top_coin"] or "n/a"),
        ]
        await update.message.reply_text("\n".join(lines))

    async def coins(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        """Show top 10 coins by historical signal performance."""
        await self._refresh_due_performance()
        rows = self.performance_tracker.coin_performance(top_n=10)

        if not rows:
            await update.message.reply_text("No signal history yet.")
            return

        header = "\U0001f3c6 Top Coins by Signal Performance\n"
        header += "\u2501" * 27
        lines = [header]
        for idx, row in enumerate(rows, start=1):
            avg_ret = row["avg_return_24h"]
            ret_str = ("{:+.2f}%".format(avg_ret)) if avg_ret is not None else "pending"
            evaluated = row["evaluated"]
            total = row["total_signals"]
            if evaluated:
                win_str = "{:.0f}% ({} eval)".format(row["win_rate"], evaluated)
            else:
                win_str = "no evals yet"
            coin_cls = get_coin_class(row["coin"])
            entry = "\n{}. {} ({})\n".format(idx, row["coin"], coin_cls)
            entry += "   Signals: {}  |  Win rate: {}\n".format(total, win_str)
            entry += "   Avg 24h return: {}  |  Avg score: {}".format(
                ret_str, row["avg_final_score"])
            lines.append(entry)

        await update.message.reply_text("\n".join(lines))
    async def mtf(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        """Show MTF diagnostic counters and first 10 failure samples."""
        d = _mtf_debug
        counts = _mtf_counts
        failures = _mtf_failures

        checked    = d.get("coins_checked", 0)
        insuff     = d.get("insufficient_history", 0)
        bull_5m    = d.get("5m_bullish", 0)
        bull_15m   = d.get("15m_bullish", 0)
        bull_1h    = d.get("1h_bullish", 0)
        full_align = d.get("full_alignment", 0)

        summary = (
            "MTF Diagnostics (this session)\n"
            "Coins checked       : " + str(checked) + "\n"
            "Insufficient history: " + str(insuff) + "\n"
            "5m bullish          : " + str(bull_5m) + "\n"
            "15m bullish         : " + str(bull_15m) + "\n"
            "1h bullish          : " + str(bull_1h) + "\n"
            "Full alignment (5m+15m+1h): " + str(full_align) + "\n"
            "\nAlignment buckets:\n"
            "  5m only    : " + str(counts.get("5m_only", 0)) + "\n"
            "  15m only   : " + str(counts.get("15m_only", 0)) + "\n"
            "  5m + 15m   : " + str(counts.get("5m_15m", 0)) + "\n"
            "  5m+15m+1h  : " + str(counts.get("5m_15m_1h", 0)) + "\n"
            "  No align   : " + str(counts.get("none", 0)) + "\n"
        )

        if failures:
            fail_lines = ["\nFirst " + str(len(failures)) + " MTF failures:"]
            for idx, f in enumerate(failures, 1):
                fail_lines.append(
                    str(idx) + ". history=" + str(f["history_len"])
                    + "  5m=" + str(f["tf_5m"])
                    + "  15m=" + str(f["tf_15m"])
                    + "  1h=" + str(f["tf_1h"])
                    + "  px=" + str(f["first_price"]) + "->" + str(f["last_price"])
                )
            summary += "\n".join(fail_lines)
        else:
            summary += "\nNo MTF failures logged yet."

        await update.message.reply_text(summary)

    async def bootstrap(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        """Show bootstrap status from the most recent startup pre-load."""
        if self.scanner is None:
            await update.message.reply_text("Scanner not initialised yet.")
            return

        br = self.scanner._bootstrap_result
        if br is None:
            await update.message.reply_text("Bootstrap has not run yet.")
            return

        # Compute live average history length from current price_history
        hist_lens = [len(v) for v in self.scanner.price_history.values() if v]
        live_avg  = sum(hist_lens) / len(hist_lens) if hist_lens else 0.0

        lines = [
            "Bootstrap Status",
            "",
            "Coins Loaded    : " + str(br.coins_loaded),
            "Failed          : " + str(br.coins_failed),
            "Avg History Len : " + "{:.1f}".format(br.avg_history_len) + " ticks (at startup)",
            "Live Avg Len    : " + "{:.1f}".format(live_avg) + " ticks (now)",
            "",
            "EMA Ready    : " + ("YES" if br.ema_ready    else "NO  (need " + str(_READY_EMA) + " ticks)"),
            "MTF Ready    : " + ("YES" if br.mtf_ready    else "NO  (need " + str(_READY_MTF_1H) + " ticks)"),
            "Phase5 Ready : " + ("YES" if br.phase5_ready else "NO  (need " + str(_READY_P5) + " ticks)"),
            "",
            "Duration     : " + "{:.1f}s".format(br.duration_s),
        ]
        if br.failed_coins:
            lines.append("")
            lines.append("Sample failures: " + ", ".join(br.failed_coins[:8]))

        await update.message.reply_text("\n".join(lines))

    async def health(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        """Report scanner + Telegram + bootstrap health."""
        br = self.scanner._bootstrap_result if self.scanner else None

        tg_ok   = _startup_diag.get("telegram_ok", False)
        bot_usr = _startup_diag.get("bot_username", "unknown")

        # Live history stats
        if self.scanner:
            hist_lens = [len(v) for v in self.scanner.price_history.values() if v]
            live_avg  = sum(hist_lens) / len(hist_lens) if hist_lens else 0.0
            coins_loaded = len(hist_lens)
        else:
            live_avg = 0.0
            coins_loaded = 0

        ema_ok    = live_avg >= _READY_EMA    if hist_lens else (br.ema_ready    if br else False)
        mtf_ok    = live_avg >= _READY_MTF_1H if hist_lens else (br.mtf_ready    if br else False)
        phase5_ok = live_avg >= _READY_P5     if hist_lens else (br.phase5_ready if br else False)

        lines = [
            "Health Check",
            "",
            "Scanner running  : " + ("YES" if self.scanner else "NO"),
            "Telegram connected: " + ("YES (@" + str(bot_usr) + ")" if tg_ok else "NO"),
            "Coins in history : " + str(coins_loaded),
            "Avg history len  : " + "{:.1f}".format(live_avg) + " ticks",
            "",
            "EMA ready    : " + ("YES" if ema_ok    else "NO (need " + str(_READY_EMA)    + " ticks)"),
            "MTF ready    : " + ("YES" if mtf_ok    else "NO (need " + str(_READY_MTF_1H) + " ticks)"),
            "Phase5 ready : " + ("YES" if phase5_ok else "NO (need " + str(_READY_P5)     + " ticks)"),
        ]
        if br:
            lines += [
                "",
                "Bootstrap duration: " + "{:.1f}s".format(br.duration_s),
                "Bootstrap loaded  : " + str(br.coins_loaded) + " coins",
            ]

        await update.message.reply_text("\n".join(lines))

    async def storage(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
        stats = self.performance_tracker.stats()
        files = self._storage_file_summary()
        lines = [
            "Storage Status",
            f"Location: {STORAGE_DIR}",
            f"Google Drive: {'yes' if USING_GOOGLE_DRIVE else 'no'}",
            f"Total signals stored: {stats['total_signals']}",
            "",
            "Files:",
        ]
        for file_info in files:
            lines.append(
                f"- {file_info['name']}: {file_info['size']} bytes, updated {file_info['updated']}"
            )
        await update.message.reply_text("\n".join(lines))

    @staticmethod
    def _storage_file_summary() -> list[dict]:
        result = []
        for name in ("signals.json", "stats.json", "watchlist.json", "scanner.log"):
            path = STORAGE_DIR / name
            if not path.exists():
                result.append({"name": name, "size": 0, "updated": "missing"})
                continue
            stat = path.stat()
            updated = datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat()
            result.append({"name": name, "size": stat.st_size, "updated": updated})
        return result
    def _register_commands(self) -> None:
        commands = {
            "start": self.start,
            "watchlist": self.show_watchlist,
            "add": self.add_coin,
            "remove": self.remove_coin,
            "scan": self.scan,
            "market": self.market,
            "top": self.top,
            "price": self.price,
            "stats": self.stats,
            "storage": self.storage,
            "coins": self.coins,
            "daily": self.daily,
            "mtf": self.mtf,
            "bootstrap": self.bootstrap,
            "health": self.health,
            "history": self.history,
            "trend": self.trend,
        }
        for command, handler in commands.items():
            self.app.add_handler(CommandHandler(command, handler))

    @staticmethod
    def _format_reasons(signal: Signal) -> str:
        return "\n".join(f"* {reason}" for reason in signal.reasons)

    @staticmethod
    def _format_signal_summary(signals: list[Signal]) -> str:
        top_signals = sorted(
            signals,
            key=lambda signal: (signal.final_score, signal.phase5_total, signal.hist_total, signal.score),
            reverse=True,
        )[:MAX_RESULTS]
        lines = [f"Top {min(MAX_RESULTS, 10)} ranked opportunities:"]
        for index, signal in enumerate(top_signals[:10], start=1):
            lines.append(
                f"\n{index}. {signal.tier} | {signal.coin}\n"
                f"Scanner:{signal.score} P5:{signal.phase5_total} Hist:{signal.hist_total} Final:{signal.final_score}\n"
                f"TQ={signal.phase5_trend} PQ={signal.phase5_pullback} MOM={signal.phase5_momentum} RR={signal.phase5_risk_reward} | 7d={signal.hist_trend_7d} 30d={signal.hist_trend_30d} 90d={signal.hist_trend_90d}\n"
                f"Price: {format_price(signal.price)}  Vol: {format_volume(signal.volume)}"
            )
        return "\n".join(lines)


# =============================================================================
# STARTUP
# =============================================================================


# =============================================================================
# STARTUP DIAGNOSTICS
# =============================================================================

_startup_diag: dict = {}   # populated by startup_diagnostics(); read by /health


async def startup_diagnostics() -> dict:
    """Verify token, reach Telegram, and log bot identity before polling starts.

    Fills the module-level _startup_diag dict so /health can report results
    without re-running network calls.
    """
    import time as _time

    diag: dict = {
        "token_loaded":    bool(BOT_TOKEN),
        "telegram_ok":     False,
        "bot_username":    None,
        "bot_id":          None,
        "webhook_cleared": False,
        "error":           None,
    }

    if not BOT_TOKEN:
        print("[Diagnostics] BOT_TOKEN is empty — cannot connect to Telegram")
        _startup_diag.update(diag)
        return diag

    print(f"[Diagnostics] Token loaded  : YES ({BOT_TOKEN[:8]}...)")

    from telegram import Bot
    from telegram.request import HTTPXRequest

    req = HTTPXRequest(
        connect_timeout = TG_CONNECT_TIMEOUT,
        read_timeout    = TG_READ_TIMEOUT,
        write_timeout   = TG_WRITE_TIMEOUT,
        pool_timeout    = TG_POOL_TIMEOUT,
    )
    bot = Bot(token=BOT_TOKEN, request=req)

    # -- getMe --
    try:
        me = await bot.get_me()
        diag["telegram_ok"]  = True
        diag["bot_username"] = me.username
        diag["bot_id"]       = me.id
        print(f"[Diagnostics] Telegram reach: YES")
        print(f"[Diagnostics] Bot username  : @{me.username}")
        print(f"[Diagnostics] Bot ID        : {me.id}")
    except Exception as exc:
        diag["error"] = str(exc)
        print(f"[Diagnostics] Telegram reach: NO  — {exc}")
        _startup_diag.update(diag)
        return diag

    # -- deleteWebhook (safe — if it times out we just continue) --
    try:
        await bot.delete_webhook(drop_pending_updates=True)
        diag["webhook_cleared"] = True
        print("[Diagnostics] Webhook       : cleared")
    except Exception as exc:
        diag["webhook_cleared"] = False
        print(f"[Diagnostics] Webhook clear : FAILED ({exc}) — continuing anyway")

    _startup_diag.update(diag)
    return diag

async def post_init(app: Application) -> None:
    scanner = app.bot_data["scanner"]
    await startup_diagnostics()
    if BOOTSTRAP_ENABLED:
        await scanner.run_bootstrap()
    app.create_task(scanner.run_forever())


def build_app() -> Application:
    watchlist = WatchlistStore()
    performance_tracker = SignalPerformanceTracker()
    telegram_bot = TelegramAlertBot(watchlist, performance_tracker)
    scanner = Scanner(watchlist, telegram_bot.send_alert, performance_tracker)

    telegram_bot.set_scanner(scanner)
    telegram_bot.app.bot_data["scanner"] = scanner
    telegram_bot.app.post_init = post_init
    return telegram_bot.app


def main() -> None:
    """Start the bot.

    Works in plain Python, Jupyter, and Google Colab.
    nest_asyncio.apply() (called at module level) allows run_polling()
    inside an already-running event loop (Colab / Jupyter).
    """
    app = build_app()
    logger.info("Crypto scanner bot starting")

    try:
        app.run_polling(
            drop_pending_updates=True,
            close_loop=False,
        )
    except Exception as exc:
        logger.exception("run_polling failed: %s", exc)
        print(f"[Startup] run_polling error: {exc}")
        raise


if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[CryptoScanner] Google Drive storage ready: /content/drive/MyDrive/CryptoScanner
[CryptoScanner] Storage: /content/drive/MyDrive/CryptoScanner  (Google Drive)
[Diagnostics] Token loaded  : YES (85041322...)
[Diagnostics] Telegram reach: YES
[Diagnostics] Bot username  : @cdxalertbot
[Diagnostics] Bot ID        : 8504132250
[Diagnostics] Webhook       : cleared


/tmp/ipykernel_114072/1315854911.py:3748: PTBUserWarning: Tasks created via `Application.create_task` while the application is not running won't be automatically awaited!
  app.create_task(scanner.run_forever())


[Bootstrap] Startup history pre-load complete
  Coins attempted : 76
  Successfully loaded : 58
  Failed          : 18
  Avg history len : 120.0 ticks
  Ready for EMA   : YES
  Ready for MTF   : YES
  Ready for Phase5: YES
  Duration        : 10.6s


ERROR:scanner_bot:Scanner loop failed; retrying next interval
Traceback (most recent call last):
  File "/tmp/ipykernel_114072/1315854911.py", line 2696, in run_forever
    watchlist_signals = await self.scan_watchlist(tickers)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114072/1315854911.py", line 2755, in scan_watchlist
    return await self._scan_many(coins, ticker_map, source="watchlist")
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114072/1315854911.py", line 2822, in _scan_many
    if not historical_filter(signal):
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114072/1315854911.py", line 1589, in historical_filter
    p90 = signal.exch_perf_90d
          ^^^^^^^^^^^^^^^^^^^^
AttributeError: 'Signal' object has no attribute 'exch_perf_90d'


Signals Returned = []
Count = 0


In [31]:
print(LATEST_MTB_SIGNALS)

[]


In [32]:
# ==========================================
# MTB Signal Bridge
# Separate from Scanner logic
# ==========================================

from datetime import datetime

LATEST_MTB_SIGNALS = []


async def get_strategy_signals(strategy="MTB"):

    if strategy != "MTB":
        return []

    return LATEST_MTB_SIGNALS


def push_mtb_signal(
        symbol,
        entry,
        stop_loss,
        take_profit,
        action="BUY"):

    global LATEST_MTB_SIGNALS

    signal = {

        "strategy": "MTB",

        "symbol": symbol,

        "action": action,

        "entry": float(entry),

        "stop_loss": float(stop_loss),

        "take_profit": float(take_profit),

        "signal_time": datetime.utcnow().isoformat() + "Z"
    }

    LATEST_MTB_SIGNALS.append(signal)

In [33]:
push_mtb_signal(
    "BTCUSDT",
    105000,
    102000,
    110000
)

/tmp/ipykernel_114072/1214725883.py:42: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "signal_time": datetime.utcnow().isoformat() + "Z"


In [ ]:
# ============================================================
# MTB API BRIDGE  —  Final cell  (do NOT modify cells above)
# Run this after main() has started the scanner.
# ============================================================

import threading
from datetime import datetime, timezone
from typing import Optional

from fastapi import FastAPI, Query
from fastapi.responses import JSONResponse
import uvicorn

# ── Shared signal list ────────────────────────────────────────
# Declared here so push_mtb_signal() can be called from any cell.
# The scanner cell already writes to this same global.
# If it was declared there first this assignment is a no-op.
try:
    LATEST_MTB_SIGNALS  # keep existing list if already defined above
except NameError:
    LATEST_MTB_SIGNALS: list[dict] = []

# ── Helper: push a signal from outside the scanner ───────────
def push_mtb_signal(
    symbol: str,
    entry: float,
    stop_loss: float,
    take_profit: float,
    action: str = "BUY",
) -> dict:
    """Append a manually-created signal to LATEST_MTB_SIGNALS.

    Args:
        symbol:      Trading pair, e.g. "BTCUSDT"
        entry:       Entry price
        stop_loss:   Stop-loss price
        take_profit: Take-profit price
        action:      "BUY" or "SELL"

    Returns the signal dict that was appended.
    """
    global LATEST_MTB_SIGNALS
    signal = {
        "strategy":    "MTB",
        "symbol":      symbol.upper(),
        "action":      action.upper(),
        "entry":       float(entry),
        "stop_loss":   float(stop_loss),
        "take_profit": float(take_profit),
        "signal_time": datetime.now(timezone.utc).isoformat(),
    }
    LATEST_MTB_SIGNALS.append(signal)
    print(f"[MTB Bridge] signal pushed: {symbol} {action} entry={entry}")
    return signal

# ── Market-state helper ───────────────────────────────────────
def _aggregate_market_state() -> dict:
    """Read the live scanner's price_history and summarise market conditions.

    Tries to use the scanner object that main() wires into bot_data.
    Falls back to a safe default if the scanner isn't running yet.
    """
    try:
        # The scanner is stored on the PTB Application via post_init
        from telegram.ext import Application
        # Reach the scanner through the global app object created by build_app()
        # build_app() returns an Application; after main() starts it lives in
        # the thread — we read _scanner directly from TelegramAlertBot instance.
        scanner_obj = None
        # Walk all live threads to find the scanner via the global telegram_bot
        import gc
        for obj in gc.get_objects():
            if type(obj).__name__ == "TelegramAlertBot" and hasattr(obj, "scanner"):
                scanner_obj = obj.scanner
                break

        if scanner_obj is None or not scanner_obj.price_history:
            return {"market": "UNKNOWN", "trend": "UNKNOWN", "coins_tracked": 0,
                    "note": "Scanner not running or no history yet"}

        ph = scanner_obj.price_history
        coins_tracked = len(ph)

        # Sample up to 20 coins and detect their market state
        state_counts: dict[str, int] = {}
        for coin, history in list(ph.items())[:20]:
            if len(history) >= 6:
                state = detect_market_state(history)
                state_counts[state] = state_counts.get(state, 0) + 1

        if not state_counts:
            dominant = "UNKNOWN"
        else:
            dominant = max(state_counts, key=state_counts.__getitem__).upper()

        return {
            "market":         dominant,
            "trend":          dominant,
            "coins_tracked":  coins_tracked,
            "state_breakdown": state_counts,
            "sampled_coins":  min(20, coins_tracked),
            "timestamp":      datetime.now(timezone.utc).isoformat(),
        }

    except Exception as exc:
        return {"market": "UNKNOWN", "trend": "UNKNOWN",
                "error": str(exc), "coins_tracked": 0}

# ── FastAPI app ───────────────────────────────────────────────
app = FastAPI(
    title="CryptoScanner MTB API",
    description="Bridge between CryptoScanner v12 and MTB trading bot",
    version="1.0.0",
)

@app.get("/api/v1/scanner/market-state")
async def get_market_state():
    """Return aggregate market state derived from live scanner price history."""
    return JSONResponse(content=_aggregate_market_state())


@app.get("/api/v1/scanner/signals")
async def get_signals(strategy: str = Query(default="MTB")):
    """Return signals filtered by strategy.

    Query param:
        strategy (str): default "MTB" — only MTB signals are stored.
    """
    if strategy.upper() != "MTB":
        return JSONResponse(content=[])
    return JSONResponse(content=LATEST_MTB_SIGNALS)


@app.get("/health")
async def health():
    return {"status": "ok", "signals": len(LATEST_MTB_SIGNALS)}


# ── Start server in background thread ────────────────────────
_API_PORT = 8000

def _run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=_API_PORT, log_level="warning")

_api_thread = threading.Thread(target=_run_uvicorn, daemon=True, name="mtb-api")
_api_thread.start()

# ── ngrok tunnel ─────────────────────────────────────────────
try:
    from pyngrok import ngrok as _ngrok
    # Auth token is already set in cell 2 — no need to set again
    _public_url = _ngrok.connect(_API_PORT)
    print(f"\n{'='*55}")
    print(f"  MTB API Bridge started")
    print(f"  Local : http://localhost:{_API_PORT}")
    print(f"  Public: {_public_url}")
    print(f"{'='*55}")
    print(f"  GET {_public_url}/api/v1/scanner/market-state")
    print(f"  GET {_public_url}/api/v1/scanner/signals?strategy=MTB")
    print(f"  GET {_public_url}/health")
    print(f"{'='*55}\n")
except Exception as _e:
    print(f"\n[MTB Bridge] ngrok not available ({_e})")
    print(f"  API running at http://localhost:{_API_PORT}")
    print(f"  Endpoints:")
    print(f"    GET /api/v1/scanner/market-state")
    print(f"    GET /api/v1/scanner/signals?strategy=MTB")
    print(f"    GET /health\n")

print("✅ MTB API Bridge ready — LATEST_MTB_SIGNALS and push_mtb_signal() available")
